# 프로젝트_소스코드_전과자

**Colored MNIST 전통 머신러닝 분류 프로젝트**

팀명: 전과자

## 주요 성능 (Validation)
| 모델 | Digit | FG | BG |
|------|-------|-----|-----|
| KNN (PCA) | 97.93% | - | - | PCA 적용 시 최고 (별도 실험) |
| XGBoost | **95.27%** | 99.99% | 100% | 종합 최고 성능 |
| SVM | 94.24% | 99.74% | 100% | 비선형 경계 학습 |
| Random Forest | 93.46% | 100% | 100% | 안정적 성능 |
| Decision Tree | 77.13% | 99.31% | 100% | 해석 가능 |
| KNN (원본) | 91.57% | 91.57% | 99.99% | 거리 기반 분류 |

## 실행 방법
1. 이 노트북을 Colab에 업로드
2. 필요한 데이터 파일을 Google Drive에 업로드
3. 셀을 순서대로 실행

**예상 실행 시간**: 약 2-3시간

In [ ]:
# ==========================================
# Colab 환경 설정
# ==========================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print('[INFO] 로컬 환경에서 실행 중')

if IN_COLAB:
    !pip install -q pyyaml xgboost opencv-python-headless

import os
from pathlib import Path

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/colored-mnist-classification')
else:
    PROJECT_ROOT = Path.cwd()
    if 'notebooks' in str(PROJECT_ROOT):
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f'[INFO] PROJECT_ROOT: {PROJECT_ROOT}')

## # PART 1: 데이터 전처리

01_preprocessing_colored_mnist.ipynb에서 가져온 코드입니다.

In [ ]:
# ======================================
# Cell 1. Imports & global config
# ======================================
# 이 셀은 노트북 실행에 필요한 모든 라이브러리를 임포트하고,
# Config 파일을 로드하는 유틸리티 함수를 정의합니다.

# ===== 표준 라이브러리 =====
import os  # 운영체제 인터페이스 (파일/디렉토리 경로 조작)
from pathlib import Path  # 경로 객체를 다루는 모던한 방법 (os.path 대체)
import yaml  # YAML 파일 파싱 (config 파일 읽기용)

# ===== 데이터 처리 라이브러리 =====
import numpy as np  # 수치 연산 및 배열 처리 (이미지 데이터 처리)
import pandas as pd  # 데이터프레임 처리 (통계 분석용)

# ===== 시각화 라이브러리 =====
import matplotlib.pyplot as plt  # 그래프/이미지 시각화

# ===== 머신러닝 라이브러리 =====
from sklearn.model_selection import train_test_split  # 데이터셋 train/val/test 분할

# ===== 이미지 처리 라이브러리 =====
from scipy.ndimage import shift, rotate, affine_transform  # 이미지 변환 (회전, 이동, 어파인 변환)
from PIL import Image, ImageDraw, ImageFont  # 이미지 생성 및 폰트 렌더링 (폰트 기반 숫자 생성용)

# ===== Matplotlib 설정 =====
plt.rcParams["figure.figsize"] = (5, 5)  # 기본 플롯 크기 설정
plt.rcParams["axes.grid"] = False  # 그리드 표시 안 함

# ===== 재현성을 위한 시드 설정 =====
RANDOM_SEED = 42  # 모든 랜덤 연산의 시드 고정 (실험 재현성 보장)
np.random.seed(RANDOM_SEED)  # NumPy 랜덤 시드 설정

# ======================================
# Config 로더 함수들
# ======================================
# 이 함수들은 YAML 형식의 config 파일을 읽어서 Python 딕셔너리로 변환합니다.
# 프로젝트의 경로와 설정을 중앙에서 관리하기 위해 사용됩니다.

def load_config(config_path):
    """
    YAML config 파일을 로드하여 dict 반환
    
    역할:
    - configs/ 디렉토리의 YAML 파일을 읽어서 Python 딕셔너리로 변환
    - 노트북이 어디서 실행되든 올바른 경로를 찾아서 config 파일 로드
    
    사용 예시:
        config = load_config("configs/paths.yaml")
        # → {'paths': {'data': {...}, 'results': {...}}}
    
    Parameters
    ----------
    config_path : str
        Config 파일 경로 (상대 또는 절대 경로)
        예: "configs/paths.yaml" 또는 "configs/models.yaml"
    
    Returns
    -------
    dict
        로드된 config 딕셔너리 (YAML 구조 그대로)
    """
    # 현재 작업 디렉토리 확인
    notebook_dir = Path.cwd()
    
    # 노트북이 notebooks/ 디렉토리 안에서 실행되는지 확인
    # → 프로젝트 루트를 자동으로 찾기 위함
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent  # notebooks/의 부모 = 프로젝트 루트
    else:
        project_root = notebook_dir  # 이미 루트에서 실행 중
    
    # Config 파일 경로 처리
    config_file = Path(config_path)
    if not config_file.is_absolute():  # 상대 경로인 경우
        config_file = project_root / config_path  # 프로젝트 루트 기준으로 변환
    
    # YAML 파일 읽기 및 파싱
    with open(config_file, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)  # 안전하게 YAML을 딕셔너리로 변환


def get_paths(config_path="configs/paths.yaml"):
    """
    경로 config를 로드하고 절대 경로로 변환
    
    역할:
    - paths.yaml에서 경로 정보를 읽어서 절대 경로 객체(Path)로 변환
    - 모든 노트북에서 일관된 경로를 사용할 수 있도록 함
    
    사용 예시:
        paths = get_paths()
        npz_path = paths["data"]["processed"]
        # → Path('/Users/.../data/processed/colored_mnist/colored_mnist_100k_train_val.npz')
    
    Parameters
    ----------
    config_path : str
        경로 config 파일 경로 (기본값: "configs/paths.yaml")
    
    Returns
    -------
    dict
        절대 경로로 변환된 경로 딕셔너리
        구조: {
            "data": {
                "raw_mnist": Path(...),
                "processed": Path(...),
                ...
            },
            "results": {
                "metrics": Path(...),
                "figures": Path(...),
                ...
            }
        }
    """
    # Config 파일 로드
    config = load_config(config_path)
    paths = config['paths']  # 'paths' 키의 값 추출
    
    # 프로젝트 루트 찾기 (load_config와 동일한 로직)
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    # 상대 경로를 절대 경로(Path 객체)로 변환
    resolved = {}
    for key, value in paths.items():
        if isinstance(value, dict):  # 중첩된 딕셔너리인 경우 (예: data, results)
            # 각 하위 경로도 절대 경로로 변환 (None 값은 그대로 유지)
            resolved[key] = {
                k: project_root / v if v is not None else None 
                for k, v in value.items()
            }
        else:  # 단일 경로인 경우
            resolved[key] = project_root / value if value is not None else None
    
    return resolved


print("[INFO] Libraries imported.")

In [ ]:
# ======================================
# Cell 2. Paths & augmentation settings (Config-based)
# ======================================
# 이 셀은 데이터 경로와 데이터 증강 설정을 정의합니다.
# 모든 경로는 configs/paths.yaml에서 중앙 관리되므로, 경로 변경 시 YAML만 수정하면 됩니다.

# ===== Config에서 경로 로드 =====
# get_paths() 함수를 사용하여 configs/paths.yaml의 모든 경로를 딕셔너리로 가져옵니다.
# 이렇게 하면 경로가 하드코딩되지 않고, 한 곳에서 관리할 수 있습니다.
paths = get_paths("configs/paths.yaml")
# paths 구조:
# {
#   "data": {
#     "raw_mnist": Path(...),
#     "raw_fonts": Path(...),
#     "processed": Path(...)
#   },
#   "results": {...}
# }

# ===== 프로젝트 루트 디렉토리 설정 =====
# 현재 작업 디렉토리를 확인하고, 프로젝트 루트를 찾습니다.
BASE_DIR = Path.cwd()
if 'notebooks' in str(BASE_DIR):
    # notebooks/ 디렉토리 안에서 실행 중이면 부모 디렉토리가 루트
    BASE_DIR = BASE_DIR.parent
# BASE_DIR은 이후에 상대 경로를 만들 때 기준이 됩니다.

# ===== 원본 데이터 경로 설정 =====
# 원본 MNIST 데이터와 폰트 파일이 저장된 경로들
RAW_DATA_DIR = BASE_DIR / "data" / "raw"  # 원본 데이터 루트 디렉토리
RAW_MNIST_DIR = BASE_DIR / "data" / "raw" / "mnist"  # MNIST 이미지가 있는 디렉토리
RAW_FONTS_DIR = paths["data"]["raw_fonts"]  # 폰트 파일(.ttf)이 있는 디렉토리 (config에서 로드)

# ===== 입력 데이터 경로 =====
# 전처리할 원본 MNIST 데이터 파일 경로
# 이 파일은 01 노트북의 입력으로 사용됩니다.
MNIST_TRAIN_PATH = paths["data"]["raw_mnist"]
# 예: data/raw/mnist/mnist_train.npz
# 구조: {'train_images': (60000, 28, 28), 'train_labels': (60000,)}

# ===== 출력 데이터 경로 설정 =====
# 전처리된 데이터를 저장할 디렉토리
PROC_DIR = BASE_DIR / "data" / "processed" / "colored_mnist"
os.makedirs(PROC_DIR, exist_ok=True)  # 디렉토리가 없으면 생성

# 최종 저장 경로: 전처리 완료된 Colored MNIST 데이터셋
# 이 파일은 02, 03 노트북의 입력으로 사용됩니다.
SAVE_PATH = paths["data"]["processed"]
# 예: data/processed/colored_mnist/colored_mnist_100k_train_val.npz
# 구조: {'X_train': (80000, 2352), 'X_val': (20000, 2352), 'y_digit_train': ...}

# ===== 데이터 증강 목표 크기 설정 =====
# 최종 데이터셋은 100K 샘플로 구성됩니다:
# - 60K: 원본 MNIST (raw)
# - 20K: Deskew 증강 (기울어진 숫자 보정)
# - 10K: Font 증강 (폰트로 생성한 숫자)
# - 10K: Geometric 증강 (회전, 이동 등)
DESKEW_AUG_TARGET = 20_000  # Deskew 증강으로 생성할 샘플 수
FONT_AUG_TARGET   = 10_000  # Font 증강으로 생성할 샘플 수
GEOM_AUG_TARGET   = 10_000  # Geometric 증강으로 생성할 샘플 수
TARGET_TOTAL_SAMPLES = 60_000 + DESKEW_AUG_TARGET + FONT_AUG_TARGET + GEOM_AUG_TARGET  # 총 100K

# ===== 정보 출력 =====
# 설정된 경로와 파라미터를 확인하기 위한 출력
print("[INFO] BASE_DIR         :", BASE_DIR)
print("[INFO] MNIST_TRAIN_PATH :", MNIST_TRAIN_PATH)
print("[INFO] RAW_FONTS_DIR    :", RAW_FONTS_DIR)
print("[INFO] PROC_DIR         :", PROC_DIR)
print("[INFO] DESKEW_AUG_TARGET:", DESKEW_AUG_TARGET)
print("[INFO] FONT_AUG_TARGET  :", FONT_AUG_TARGET)
print("[INFO] GEOM_AUG_TARGET  :", GEOM_AUG_TARGET)

# ===== 색상 팔레트 정의 =====
# RAINBOW (ROYGBIV) 7색을 RGB 값으로 정의
# 이 색상들은 전경색(foreground)과 배경색(background)으로 사용됩니다.
COLOR_PALETTE = np.array([
    [255,   0,   0],  # Red (빨강)
    [255, 144,   0],  # Orange (주황)
    [255, 220,   0],  # Yellow (노랑)
    [  0, 255,   0],  # Green (초록)
    [  0,   0, 255],  # Blue (파랑)
    [ 25,   0, 120],  # Indigo (남색)
    [110,   0, 180],  # Violet (보라)
], dtype=np.uint8)  # 0-255 범위의 정수형 배열

# ===== 이미지 크기 상수 =====
N_COLORS   = COLOR_PALETTE.shape[0]  # 색상 개수: 7
IMG_HEIGHT = 28  # MNIST 이미지 높이 (픽셀)
IMG_WIDTH  = 28  # MNIST 이미지 너비 (픽셀)
# MNIST는 28x28 그레이스케일 이미지입니다.
# Colored MNIST로 변환하면 28x28x3 (RGB) 이미지가 됩니다.

In [ ]:
# ======================================
# Cell 3. Load raw MNIST (train_images / train_labels) + basic EDA
# ======================================
# 이 셀은 원본 MNIST 데이터를 로드하고, 데이터의 기본 통계를 확인합니다.
# EDA(Exploratory Data Analysis)를 통해 데이터의 분포와 특성을 파악합니다.

def load_mnist_npz(npz_path: str):
    """
    원본 MNIST 데이터를 npz 파일에서 로드하는 함수
    
    역할:
    - npz 파일에서 train_images와 train_labels를 읽어옵니다
    - 데이터 타입을 float32와 int64로 변환합니다
    - 데이터 형식이 올바른지 검증합니다
    
    Parameters
    ----------
    npz_path : str
        MNIST npz 파일 경로
        예: "data/raw/mnist/mnist_train.npz"
    
    Returns
    -------
    X : np.ndarray
        이미지 데이터 (N, 28, 28), float32 타입
        N은 샘플 수 (일반적으로 60000)
    y : np.ndarray
        라벨 데이터 (N,), int64 타입
        각 샘플의 숫자 라벨 (0-9)
    
    Raises
    ------
    ValueError
        - 필수 키가 없거나
        - 이미지 형식이 (N, 28, 28)이 아닌 경우
    """
    # npz 파일 로드 (NumPy의 압축된 배열 형식)
    data = np.load(npz_path)
    print("[INFO] Keys in mnist_train.npz:", data.files)

    # 필수 키 존재 확인
    if "train_images" not in data.files or "train_labels" not in data.files:
        raise ValueError("Expected keys 'train_images' and 'train_labels' in mnist_train.npz.")

    # 이미지 데이터 로드 및 타입 변환
    # uint8 (0-255) → float32 (계산 효율성 및 정밀도 향상)
    X = data["train_images"].astype(np.float32)  # (N, 28, 28)
    
    # 라벨 데이터 로드 및 타입 변환
    y = data["train_labels"].astype(np.int64)   # (N,)
    
    # 이미지 형식 검증: 3차원 배열이고, 각 이미지가 28x28인지 확인
    if X.ndim != 3 or X.shape[1:] != (28, 28):
        raise ValueError(f"Unexpected train_images shape: {X.shape}")

    return X, y


# ===== 원본 MNIST 데이터 로드 =====
# load_mnist_npz() 함수를 사용하여 원본 MNIST 데이터를 로드합니다.
X_raw, y_raw = load_mnist_npz(MNIST_TRAIN_PATH)

# ===== 기본 정보 확인 =====
N_BASE = X_raw.shape[0]  # 원본 샘플 수 (일반적으로 60000)
print(f"[INFO] Raw MNIST shape (N, H, W): {X_raw.shape}")
print(f"[INFO] Raw labels shape         : {y_raw.shape}")
print(f"[INFO] Unique labels            : {np.unique(y_raw)}")

# ===== EDA 1: 라벨 분포 확인 =====
# 각 숫자(0-9)가 데이터셋에 몇 개씩 있는지 확인합니다.
# 클래스 불균형을 확인하기 위한 중요한 단계입니다.
label_counts = pd.Series(y_raw).value_counts().sort_index()
print("\n[EDA] Digit label counts (raw):")
print(label_counts)

# 라벨 분포를 막대 그래프로 시각화
plt.figure()
label_counts.plot(kind="bar")
plt.title("Raw MNIST - Digit Label Distribution")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

# ===== EDA 2: 픽셀 통계 확인 =====
# 이미지 픽셀 값의 분포를 확인합니다.
# - min/max: 픽셀 값의 범위 (일반적으로 0-255)
# - mean/std: 픽셀 값의 평균과 표준편차 (데이터 정규화 시 필요)
flat_pixels = X_raw.reshape(-1)  # 모든 이미지를 1차원으로 펼침
print(f"[EDA] Pixel min/max (raw): {flat_pixels.min()} / {flat_pixels.max()}")
print(f"[EDA] Pixel mean/std (raw): {flat_pixels.mean():.2f} / {flat_pixels.std():.2f}")

# 픽셀 값 분포를 히스토그램으로 시각화
plt.figure()
plt.hist(flat_pixels, bins=30, range=(0, 255))
plt.title("Raw MNIST - Pixel Intensity Histogram")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.show()

# ===== EDA 3: 클래스별 평균 이미지 =====
# 각 숫자 클래스의 평균 이미지를 시각화합니다.
# 이를 통해 각 숫자의 특징적인 패턴을 확인할 수 있습니다.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
fig.suptitle("Raw MNIST - Mean Image per Digit", fontsize=12)
for d in range(10):  # 0부터 9까지
    # 해당 숫자의 모든 이미지의 평균 계산
    mean_img = X_raw[y_raw == d].mean(axis=0)
    ax = axes[d // 5, d % 5]  # 2x5 그리드에 배치
    ax.imshow(mean_img, cmap="gray")
    ax.set_title(f"Digit {d}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# ===== EDA 4: 랜덤 샘플 시각화 =====
# 데이터셋에서 랜덤으로 선택한 샘플들을 시각화합니다.
# 데이터가 올바르게 로드되었는지 확인하는 데 사용됩니다.

def plot_random_digits(X, y, n=16, title="Random Raw Digits"):
    """
    데이터셋에서 랜덤으로 선택한 숫자 이미지를 시각화하는 함수
    
    역할:
    - 데이터셋에서 n개의 랜덤 샘플을 선택하여 그리드 형태로 표시
    - 각 이미지 위에 라벨을 표시하여 데이터 확인 용이
    
    Parameters
    ----------
    X : np.ndarray
        이미지 배열 (N, 28, 28)
    y : np.ndarray
        라벨 배열 (N,)
    n : int
        표시할 샘플 수 (기본값: 16)
    title : str
        그래프 제목
    """
    # 중복 없이 n개의 랜덤 인덱스 선택
    idx = np.random.choice(len(X), size=n, replace=False)
    cols = 4  # 그리드 열 개수
    rows = int(np.ceil(n / cols))  # 그리드 행 개수
    plt.figure(figsize=(cols * 2, rows * 2))
    for i, j in enumerate(idx):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(X[j], cmap="gray")  # 그레이스케일 이미지 표시
        plt.title(f"y={y[j]}")  # 라벨 표시
        plt.axis("off")  # 축 제거
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_random_digits(X_raw, y_raw, n=16, title="Raw MNIST - Random Samples")

In [ ]:
# ======================================
# Cell 4. Deskew augmentation (+20k)
# ======================================
# 이 셀은 Deskew(기울어진 숫자 보정) 증강을 수행합니다.
# 기울어진 숫자를 수직으로 보정하여 클래스 내 분산을 줄이고 모델 성능을 향상시킵니다.
# 목표: 원본 MNIST에서 20,000개의 Deskew 증강 샘플 생성

def deskew_image(image: np.ndarray) -> np.ndarray:
    """
    단일 28x28 이미지의 기울기를 보정하는 함수 (Deskew)
    
    역할:
    - 이미지 모멘트(image moments)를 사용하여 숫자의 기울기를 계산
    - 어파인 변환(affine transform)을 적용하여 숫자를 수직으로 보정
    - 기울어진 숫자로 인한 클래스 내 분산을 줄여 모델 학습에 도움
    
    원리:
    1. 이미지의 중심(centroid) 계산
    2. 기울기 계수(skew factor) 계산: mu11 / mu20
    3. 어파인 변환 행렬 생성 및 적용
    
    Parameters
    ----------
    image : np.ndarray
        입력 이미지 (28, 28), 그레이스케일
    
    Returns
    -------
    np.ndarray
        보정된 이미지 (28, 28), 원본과 동일한 dtype
    """
    # 계산 정밀도를 위해 float64로 변환
    img = image.astype(np.float64)
    h, w = img.shape  # 이미지 크기 (28, 28)

    # 좌표 그리드 생성 (각 픽셀의 y, x 좌표)
    c0, c1 = np.mgrid[0:h, 0:w]
    
    # 이미지의 총 픽셀 값 합계 (정규화에 사용)
    total = img.sum()
    if total == 0:  # 빈 이미지인 경우 원본 반환
        return image

    # 1차 모멘트: 이미지의 중심(centroid) 계산
    m0 = (c0 * img).sum() / total  # y 방향 중심
    m1 = (c1 * img).sum() / total  # x 방향 중심

    # 2차 중심 모멘트: 기울기 계산에 사용
    mu11 = ((c0 - m0) * (c1 - m1) * img).sum() / total  # 교차 모멘트
    mu20 = ((c0 - m0) ** 2 * img).sum() / total  # y 방향 분산

    if mu20 == 0:  # 분산이 0이면 기울기 없음 → 원본 반환
        return image

    # 기울기 계수 계산: mu11 / mu20
    # alpha가 0에 가까우면 수직, 크면 기울어짐
    alpha = mu11 / mu20  # skew factor

    # 어파인 변환 행렬 생성
    # [[1, 0], [alpha, 1]]: y 방향으로 alpha만큼 기울기 보정
    affine = np.array([[1, 0], [alpha, 1]])
    center = np.array([h / 2.0, w / 2.0])  # 이미지 중심점
    offset = center - affine @ center  # 중심점 유지를 위한 오프셋

    # 어파인 변환 적용하여 기울기 보정
    deskewed = affine_transform(
        img,
        affine,
        offset=offset,
        order=1,  # 선형 보간
        mode="constant",  # 경계 처리: 상수값(0)으로 채움
        cval=0.0,  # 경계 채움 값
    )

    # 원본 dtype으로 변환하여 반환
    return deskewed.astype(image.dtype)


def augment_deskew(X: np.ndarray,
                   y: np.ndarray,
                   n_aug: int,
                   random_state: int = 101):
    """
    원본 이미지에서 랜덤으로 샘플링하여 Deskew 증강을 수행하는 함수
    
    역할:
    - 원본 데이터셋에서 n_aug개의 이미지를 랜덤으로 선택
    - 각 이미지에 deskew_image() 함수를 적용하여 기울기 보정
    - 증강된 이미지와 라벨, 원본 인덱스를 반환
    
    Parameters
    ----------
    X : np.ndarray
        원본 이미지 배열 (N, 28, 28)
    y : np.ndarray
        원본 라벨 배열 (N,)
    n_aug : int
        생성할 증강 샘플 수 (예: 20000)
    random_state : int
        랜덤 시드 (재현성 보장)
    
    Returns
    -------
    X_aug : np.ndarray
        증강된 이미지 배열 (n_aug, 28, 28)
    y_aug : np.ndarray
        증강된 이미지의 라벨 배열 (n_aug,)
    idx : np.ndarray
        원본 이미지의 인덱스 배열 (n_aug,)
        어떤 원본 이미지에서 증강되었는지 추적
    """
    # 랜덤 생성기 초기화 (재현성 보장)
    rng = np.random.default_rng(random_state)
    N = len(X)
    
    # 원본 데이터셋에서 n_aug개의 랜덤 인덱스 선택 (중복 허용 가능)
    idx = rng.integers(0, N, size=n_aug)

    # 증강된 이미지를 저장할 배열 초기화
    X_aug = np.empty((n_aug, IMG_HEIGHT, IMG_WIDTH), dtype=X.dtype)
    y_aug = y[idx].copy()  # 선택된 이미지의 라벨 복사

    # 각 선택된 이미지에 대해 Deskew 적용
    for i, j in enumerate(idx):
        X_aug[i] = deskew_image(X[j])

    return X_aug, y_aug, idx


# ===== Deskew 증강 실행 =====
# 원본 MNIST에서 DESKEW_AUG_TARGET(20000)개의 샘플을 랜덤 선택하여 Deskew 적용
X_deskew, y_deskew, idx_deskew = augment_deskew(
    X_raw, y_raw, n_aug=DESKEW_AUG_TARGET, random_state=101
)
print("[INFO] Deskew-augmented shape:", X_deskew.shape)

# ===== EDA: 원본 vs Deskew 픽셀 통계 비교 =====
# Deskew 증강이 픽셀 분포에 미치는 영향을 확인합니다.
flat_raw    = X_raw.reshape(-1)      # 원본 이미지의 모든 픽셀을 1차원으로
flat_deskew = X_deskew.reshape(-1)   # Deskew 이미지의 모든 픽셀을 1차원으로

print("\n[EDA] Pixel mean/std (raw)       : "
      f"{flat_raw.mean():.2f} / {flat_raw.std():.2f}")
print("[EDA] Pixel mean/std (deskew_aug): "
      f"{flat_deskew.mean():.2f} / {flat_deskew.std():.2f}")

# 원본과 Deskew의 픽셀 분포를 히스토그램으로 비교
plt.figure()
plt.hist(flat_raw,    bins=30, range=(0, 255), alpha=0.5, label="raw")
plt.hist(flat_deskew, bins=30, range=(0, 255), alpha=0.5, label="deskew")
plt.title("Pixel Histogram - Raw vs Deskew Augmentation")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# ===== EDA: 원본 vs Deskew 시각적 비교 =====
# 원본 이미지와 Deskew 적용 후 이미지를 나란히 비교하여 효과를 확인합니다.

def plot_deskew_examples(X_original, X_aug, idx_original, y, n=8):
    """
    원본 이미지와 Deskew 적용 후 이미지를 비교하여 시각화하는 함수
    
    역할:
    - 증강된 이미지 중 n개를 랜덤 선택
    - 각 이미지의 원본과 Deskew 버전을 위아래로 나란히 표시
    - Deskew 효과를 시각적으로 확인
    
    Parameters
    ----------
    X_original : np.ndarray
        원본 이미지 배열
    X_aug : np.ndarray
        증강된 이미지 배열
    idx_original : np.ndarray
        원본 이미지 인덱스 배열
    y : np.ndarray
        라벨 배열
    n : int
        비교할 샘플 수 (기본값: 8)
    """
    # 증강된 이미지 중 n개를 랜덤 선택
    sample_idx = np.random.choice(len(X_aug), size=n, replace=False)
    cols = n  # 가로로 n개 배치
    rows = 2  # 세로로 2줄 (원본, Deskew)
    plt.figure(figsize=(cols * 2, rows * 2))
    for k, i in enumerate(sample_idx):
        orig_index = idx_original[i]  # 원본 이미지 인덱스

        # 원본 이미지 표시 (위쪽 행)
        plt.subplot(rows, cols, k + 1)
        plt.imshow(X_original[orig_index], cmap="gray")
        plt.title(f"Raw (y={y[orig_index]})")
        plt.axis("off")

        # Deskew 이미지 표시 (아래쪽 행)
        plt.subplot(rows, cols, k + 1 + cols)
        plt.imshow(X_aug[i], cmap="gray")
        plt.title("Deskew")
        plt.axis("off")

    plt.suptitle("Deskew Augmentation - Raw vs Deskewed")
    plt.tight_layout()
    plt.show()

plot_deskew_examples(X_raw, X_deskew, idx_deskew, y_raw, n=8)



In [ ]:
# ======================================
# Cell 5. Font-based synthetic (+10k)
# ======================================
# 이 셀은 폰트를 사용하여 합성 숫자 이미지를 생성합니다.
# TrueType 폰트를 사용하여 0-9 숫자를 렌더링하고, 이를 28x28 이미지로 변환합니다.
# 목표: 10,000개의 폰트 기반 합성 샘플 생성

# ===== 사용할 폰트 파일 경로 목록 =====
# 여러 폰트를 사용하여 다양한 스타일의 숫자를 생성합니다.
# 각 폰트는 다른 굵기(ExtraLight, Light, Regular, SemiBold, Bold)를 제공합니다.
FONT_PATHS = [
    os.path.join(RAW_FONTS_DIR, "MaruBuri-ExtraLight.ttf"),
    os.path.join(RAW_FONTS_DIR, "MaruBuri-Light.ttf"),
    os.path.join(RAW_FONTS_DIR, "MaruBuri-Regular.ttf"),
    os.path.join(RAW_FONTS_DIR, "MaruBuri-SemiBold.ttf"),
    os.path.join(RAW_FONTS_DIR, "MaruBuri-Bold.ttf"),
]

def render_digit_with_font(digit: int,
                           font_path: str,
                           img_size: int = 28) -> np.ndarray:
    """
    TrueType 폰트를 사용하여 단일 숫자를 렌더링하고 28x28 캔버스에 중앙 정렬하는 함수
    
    역할:
    - 폰트를 사용하여 숫자를 이미지로 렌더링
    - 숫자를 28x28 크기로 정확히 중앙 정렬
    - baseline bias 제거 (숫자가 아래로 치우치지 않도록)
    
    처리 과정:
    1. 큰 캔버스(64x64)에 숫자 그리기
    2. 실제 그려진 픽셀의 tight bounding box 찾기
    3. 해당 영역만 잘라내기
    4. 여백을 두고 리사이즈한 후 28x28 캔버스 중앙에 배치
    
    Parameters
    ----------
    digit : int
        렌더링할 숫자 (0-9)
    font_path : str
        TrueType 폰트 파일 경로 (.ttf)
    img_size : int
        최종 이미지 크기 (기본값: 28)
    
    Returns
    -------
    np.ndarray
        렌더링된 숫자 이미지 (28, 28), float32 타입
    """
    # ===== 1단계: 큰 캔버스에 숫자 그리기 =====
    # 큰 캔버스를 사용하여 숫자가 잘리지 않도록 합니다.
    BIG_SIZE = 64  # 임시 캔버스 크기 (최종 28x28보다 큼)
    img_big = Image.new("L", (BIG_SIZE, BIG_SIZE), color=0)  # 검은 배경
    draw_big = ImageDraw.Draw(img_big)

    # 폰트 선택 및 크기 설정 (최종 크기보다 크게 설정)
    font_size = 48  # 최종 28x28보다 큰 폰트 크기
    font = ImageFont.truetype(font_path, font_size)
    text = str(digit)  # 숫자를 문자열로 변환

    # 큰 캔버스에 대략적으로 중앙 정렬
    # textbbox 사용 (Pillow >= 8.0, deprecated textsize 대신)
    bbox = draw_big.textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0]  # 텍스트 너비
    h = bbox[3] - bbox[1]  # 텍스트 높이
    x = (BIG_SIZE - w) // 2  # 중앙 정렬 x 좌표
    y = (BIG_SIZE - h) // 2  # 중앙 정렬 y 좌표
    draw_big.text((x, y), text, fill=255, font=font)  # 흰색(255)으로 숫자 그리기

    # ===== 2단계: 실제 그려진 픽셀의 tight bounding box 찾기 =====
    # 배열로 변환하여 0이 아닌 픽셀의 위치 찾기
    arr_big = np.array(img_big, dtype=np.float32)
    ys, xs = np.where(arr_big > 0)  # 0보다 큰 픽셀의 좌표

    if len(xs) == 0 or len(ys) == 0:
        # 렌더링 실패 시 빈 이미지 반환
        return np.zeros((img_size, img_size), dtype=np.float32)

    # tight bounding box 계산
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    # ===== 3단계: tight bounding box만 잘라내기 =====
    crop = arr_big[y_min:y_max + 1, x_min:x_max + 1]  # (Hc, Wc)

    # ===== 4단계: 여백을 두고 리사이즈한 후 28x28 캔버스 중앙에 배치 =====
    h_crop, w_crop = crop.shape
    # 각 면에 2픽셀 여백을 두므로 (img_size-4) 크기로 맞춤
    target_side = img_size - 4
    # 가로세로 비율을 유지하면서 target_side에 맞추기 위한 스케일 계산
    scale = min(target_side / h_crop, target_side / w_crop)
    new_w = max(1, int(round(w_crop * scale)))
    new_h = max(1, int(round(h_crop * scale)))

    # PIL Image로 변환하여 리사이즈
    crop_img = Image.fromarray(crop)
    crop_resized = crop_img.resize((new_w, new_h), resample=Image.BILINEAR)  # 양선형 보간

    # ===== 최종: 리사이즈된 숫자를 28x28 캔버스 중앙에 배치 =====
    canvas = Image.new("L", (img_size, img_size), color=0)  # 검은 배경 캔버스
    x_off = (img_size - new_w) // 2  # 중앙 정렬 x 오프셋
    y_off = (img_size - new_h) // 2  # 중앙 정렬 y 오프셋
    canvas.paste(crop_resized, (x_off, y_off))  # 중앙에 붙여넣기

    return np.array(canvas, dtype=np.float32)


def generate_font_synthetic(n_samples: int,
                            font_paths,
                            random_state: int = 202):
    """
    여러 폰트를 사용하여 합성 숫자 이미지를 생성하는 함수
    
    역할:
    - 0-9 숫자를 균등하게 샘플링
    - 각 숫자에 대해 랜덤으로 폰트 선택
    - render_digit_with_font()를 사용하여 중앙 정렬된 이미지 생성
    
    Parameters
    ----------
    n_samples : int
        생성할 샘플 수 (예: 10000)
    font_paths : list
        사용할 폰트 파일 경로 리스트
    random_state : int
        랜덤 시드 (재현성 보장)
    
    Returns
    -------
    X_syn : np.ndarray
        합성 이미지 배열 (n_samples, 28, 28), float32
    y_syn : np.ndarray
        숫자 라벨 배열 (n_samples,), int64
    """
    # 랜덤 생성기 초기화
    rng = np.random.default_rng(random_state)
    # 0-9 숫자를 균등하게 n_samples개 샘플링
    digits = rng.integers(0, 10, size=n_samples)

    # 합성 이미지를 저장할 배열 초기화
    X_syn = np.empty((n_samples, IMG_HEIGHT, IMG_WIDTH), dtype=np.float32)
    y_syn = digits.astype(np.int64)

    # 각 샘플에 대해 폰트 기반 숫자 렌더링
    for i in range(n_samples):
        d = int(digits[i])  # 렌더링할 숫자
        font_path = rng.choice(font_paths)  # 랜덤 폰트 선택
        X_syn[i] = render_digit_with_font(d, font_path, img_size=IMG_HEIGHT)

    return X_syn, y_syn


# ===== 폰트 기반 합성 샘플 생성 =====
# FONT_AUG_TARGET(10000)개의 폰트 기반 합성 샘플 생성
X_font, y_font = generate_font_synthetic(
    n_samples=FONT_AUG_TARGET,
    font_paths=FONT_PATHS,
    random_state=202,
)
print("[INFO] Font synthetic shape:", X_font.shape)

# ===== EDA: 폰트 합성 샘플의 라벨 분포 및 시각화 =====
# 생성된 샘플의 숫자 분포가 균등한지 확인합니다.
font_label_counts = pd.Series(y_font).value_counts().sort_index()
print("\n[EDA] Digit label counts (font synthetic):")
print(font_label_counts)

# 라벨 분포를 막대 그래프로 시각화
plt.figure()
font_label_counts.plot(kind="bar")
plt.title("Font Synthetic - Digit Label Distribution")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

def plot_font_examples(X, y, n=12):
    """
    폰트 기반 합성 숫자 이미지를 시각화하는 함수
    
    역할:
    - 생성된 합성 샘플 중 n개를 랜덤 선택하여 표시
    - 숫자가 수직/수평으로 중앙 정렬되었는지 확인
    
    Parameters
    ----------
    X : np.ndarray
        합성 이미지 배열
    y : np.ndarray
        라벨 배열
    n : int
        표시할 샘플 수 (기본값: 12)
    """
    idx = np.random.choice(len(X), size=n, replace=False)
    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(cols * 2, rows * 2))
    for k, i in enumerate(idx):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(X[i], cmap="gray")
        plt.title(f"y={y[i]}")
        plt.axis("off")
    plt.suptitle("Font-based Synthetic Digits (Centered)")
    plt.tight_layout()
    plt.show()

plot_font_examples(X_font, y_font, n=12)

In [ ]:
# ======================================
# Cell 6. Shift + rotation augmentation (+10k)
# ======================================
# 이 셀은 기하학적 변환(이동, 회전)을 사용한 데이터 증강을 수행합니다.
# 원본 MNIST 이미지에 랜덤 이동과 회전을 적용하여 다양한 변형을 생성합니다.
# 목표: 원본 MNIST에서 10,000개의 기하학적 증강 샘플 생성

from scipy.ndimage import shift, rotate  # 다시 한 번 확실히 임포트

def augment_shift_rotation(
    X: np.ndarray,
    y: np.ndarray,
    n_aug: int,
    max_shift: int = 2,
    max_rot: float = 15.0,
    random_state: int = 303,
):
    """
    랜덤 이동과 회전을 적용하여 증강 이미지를 생성하는 함수
    
    역할:
    - 원본 이미지에 랜덤 이동(shift)과 회전(rotation) 적용
    - 모델이 다양한 위치와 각도의 숫자에 강건하도록 학습 데이터 확장
    - 데이터 증강을 통한 일반화 성능 향상
    
    Parameters
    ----------
    X : np.ndarray
        원본 이미지 배열 (N, 28, 28), 그레이스케일
    y : np.ndarray
        원본 라벨 배열 (N,)
    n_aug : int
        생성할 증강 샘플 수 (예: 10000)
    max_shift : int
        x/y 방향 최대 픽셀 이동량 (±max_shift)
        예: max_shift=2 → -2 ~ +2 픽셀 범위에서 이동
    max_rot : float
        최대 회전 각도 (도 단위)
        예: max_rot=15.0 → -15° ~ +15° 범위에서 회전
    random_state : int
        랜덤 시드 (재현성 보장)
    
    Returns
    -------
    X_aug : np.ndarray
        증강된 이미지 배열 (n_aug, 28, 28), float32
    y_aug : np.ndarray
        증강된 이미지의 라벨 배열 (n_aug,), int64
    """
    # 랜덤 생성기 초기화
    rng = np.random.default_rng(random_state)
    N = len(X)

    # 증강된 이미지와 라벨을 저장할 배열 초기화
    X_aug = np.empty((n_aug, IMG_HEIGHT, IMG_WIDTH), dtype=np.float32)
    y_aug = np.empty((n_aug,), dtype=np.int64)

    # 각 증강 샘플 생성
    for i in range(n_aug):
        # 원본 데이터셋에서 랜덤으로 이미지 선택
        idx = rng.integers(0, N)
        img = X[idx]

        # ===== 랜덤 이동(Shift) 적용 =====
        # y 방향과 x 방향으로 각각 랜덤 이동
        dy = rng.integers(-max_shift, max_shift + 1)  # y 방향 이동 (-2 ~ +2)
        dx = rng.integers(-max_shift, max_shift + 1)  # x 방향 이동 (-2 ~ +2)
        # shift 함수: 이미지를 (dy, dx)만큼 이동, 빈 공간은 0으로 채움
        shifted = shift(img, shift=(dy, dx), cval=0.0)

        # ===== 랜덤 회전(Rotation) 적용 =====
        # -max_rot ~ +max_rot 범위에서 균등 분포로 각도 선택
        angle = rng.uniform(-max_rot, max_rot)
        # rotate 함수: 이미지를 angle도만큼 회전
        # reshape=False: 이미지 크기 유지 (28x28)
        # cval=0.0: 회전으로 생긴 빈 공간은 0으로 채움
        # order=1: 선형 보간 사용
        rotated = rotate(
            shifted,
            angle=angle,
            reshape=False,
            cval=0.0,
            order=1,
        )

        # 증강된 이미지와 라벨 저장
        X_aug[i] = rotated
        y_aug[i] = y[idx]

    return X_aug, y_aug


# ===== 기하학적 증강 샘플 생성 =====
# 원본 MNIST에서 GEOM_AUG_TARGET(10000)개의 샘플을 랜덤 선택하여
# 이동과 회전을 적용한 증강 샘플 생성
X_geom, y_geom = augment_shift_rotation(
    X_raw,          # 원본 MNIST에서 샘플링
    y_raw,
    n_aug=GEOM_AUG_TARGET,
    max_shift=2,      # 최대 ±2픽셀 이동
    max_rot=15.0,     # 최대 ±15도 회전
    random_state=303,
)

print("[INFO] Geometric augmented shape:", X_geom.shape)

# ===== EDA: 기하학적 증강 샘플의 라벨 분포 및 시각화 =====
# 생성된 샘플의 숫자 분포를 확인합니다.
geom_label_counts = pd.Series(y_geom).value_counts().sort_index()
print("\n[EDA] Digit label counts (geom augmentation):")
print(geom_label_counts)

# 라벨 분포를 막대 그래프로 시각화
plt.figure()
geom_label_counts.plot(kind="bar")
plt.title("Geom Augmentation - Digit Label Distribution")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

def plot_geom_examples(X_aug, y_aug, n=12):
    """
    이동+회전 증강된 숫자 이미지를 시각화하는 함수
    
    역할:
    - 증강된 샘플 중 n개를 랜덤 선택하여 표시
    - 이동과 회전이 올바르게 적용되었는지 확인
    
    Parameters
    ----------
    X_aug : np.ndarray
        증강된 이미지 배열
    y_aug : np.ndarray
        라벨 배열
    n : int
        표시할 샘플 수 (기본값: 12)
    """
    idx = np.random.choice(len(X_aug), size=n, replace=False)
    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(cols * 2, rows * 2))
    for k, i in enumerate(idx):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(X_aug[i], cmap="gray")
        plt.title(f"y={y_aug[i]}")
        plt.axis("off")
    plt.suptitle("Shift + Rotation Augmentation")
    plt.tight_layout()
    plt.show()

plot_geom_examples(X_geom, y_geom, n=12)

In [ ]:
# ======================================
# Cell 7. Merge all sources → 100k
# ======================================
# 이 셀은 모든 증강 소스(원본, Deskew, Font, Geometric)를 하나로 병합합니다.
# 최종적으로 100,000개의 샘플을 생성합니다:
# - 60,000: 원본 MNIST (raw)
# - 20,000: Deskew 증강
# - 10,000: Font 증강
# - 10,000: Geometric 증강

# ===== 안전성 검사: 모든 증강 배열이 존재하는지 확인 =====
# 이전 셀들(Cell 3~6)이 순서대로 실행되었는지 확인합니다.
for name in ["X_raw", "X_deskew", "X_font", "X_geom",
             "y_raw", "y_deskew", "y_font", "y_geom"]:
    if name not in globals():
        raise RuntimeError(f"[ERROR] {name} is not defined. "
                           f"Run Cells 3~6 in order before this cell.")

# ===== 데이터 타입 통일 =====
# 모든 이미지를 float32로 변환하여 일관성 유지
X_raw    = X_raw.astype(np.float32)
X_deskew = X_deskew.astype(np.float32)
X_font   = X_font.astype(np.float32)
X_geom   = X_geom.astype(np.float32)

# ===== 모든 소스 병합 =====
# 이미지와 라벨을 각각 첫 번째 차원(axis=0)을 따라 연결
X_all = np.concatenate([X_raw, X_deskew, X_font, X_geom], axis=0)
y_all = np.concatenate([y_raw, y_deskew, y_font, y_geom], axis=0)

# ===== 각 샘플의 출처 추적 =====
# 각 샘플이 어떤 증강 방법으로 생성되었는지 기록
# 이후 오분류 분석 시 어떤 증강 방법이 어려운지 확인 가능
source_tags = (
    ["raw"]    * len(X_raw)    +  # 원본: 60,000개
    ["deskew"] * len(X_deskew) +  # Deskew: 20,000개
    ["font"]   * len(X_font)   +  # Font: 10,000개
    ["geom"]   * len(X_geom)       # Geometric: 10,000개
)
source_all = np.array(source_tags)  # 문자열 배열로 변환

# ===== 병합 결과 확인 =====
print("[INFO] Total digits shape :", X_all.shape)
print("[INFO] Total labels shape :", y_all.shape)
print("[INFO] Source array shape :", source_all.shape)

# ===== 일관성 검증 =====
# 총 샘플 수가 목표(TARGET_TOTAL_SAMPLES = 100,000)와 일치하는지 확인
assert X_all.shape[0] == TARGET_TOTAL_SAMPLES, \
    f"Expected {TARGET_TOTAL_SAMPLES}, got {X_all.shape[0]}"
assert y_all.shape[0] == TARGET_TOTAL_SAMPLES
assert source_all.shape[0] == TARGET_TOTAL_SAMPLES

# ===== EDA: 전체 라벨 및 출처 분포 확인 =====

# 숫자 라벨 분포 확인
label_counts_all = pd.Series(y_all).value_counts().sort_index()
print("\n[EDA] Digit label counts (all samples):")
print(label_counts_all)

# 숫자 라벨 분포를 막대 그래프로 시각화
plt.figure()
label_counts_all.plot(kind="bar")
plt.title("All Samples - Digit Label Distribution")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

# 출처(증강 방법) 분포 확인
source_counts = pd.Series(source_all).value_counts()
print("\n[EDA] Source distribution (raw / deskew / font / geom):")
print(source_counts)

# 출처 분포를 막대 그래프로 시각화
plt.figure()
source_counts.plot(kind="bar")
plt.title("All Samples - Source Distribution")
plt.xlabel("Source")
plt.ylabel("Count")
plt.show()

# ===== 교차표: 숫자 vs 출처 =====
# 각 숫자별로 어떤 증강 방법으로 생성되었는지 확인
crosstab_source = pd.crosstab(y_all, source_all)
print("\n[EDA] Crosstab (digit vs source):")
print(crosstab_source)

In [ ]:
# ======================================
# Cell 8. fg/bg color labels + colorize
# ======================================
# 이 셀은 각 샘플에 전경색(foreground)과 배경색(background) 라벨을 할당하고,
# 그레이스케일 이미지를 컬러 이미지로 변환합니다.
# 전경색과 배경색은 항상 다르게 할당됩니다 (fg != bg).

def assign_fg_bg_labels(n_samples: int,
                        n_colors: int,
                        random_state: int = 777):
    """
    전경색과 배경색 라벨을 랜덤으로 할당하는 함수
    
    역할:
    - 각 샘플에 대해 전경색(숫자 색상)과 배경색 라벨을 랜덤 할당
    - 전경색과 배경색이 같지 않도록 보장 (fg != bg)
    - 7가지 색상(ROYGBIV) 중에서 선택
    
    Parameters
    ----------
    n_samples : int
        샘플 수 (예: 100000)
    n_colors : int
        사용 가능한 색상 개수 (7: ROYGBIV)
    random_state : int
        랜덤 시드 (재현성 보장)
    
    Returns
    -------
    fg : np.ndarray
        전경색 라벨 배열 (n_samples,), int64
        각 값은 0-6 범위 (ROYGBIV)
    bg : np.ndarray
        배경색 라벨 배열 (n_samples,), int64
        각 값은 0-6 범위 (ROYGBIV)
    """
    # 랜덤 생성기 초기화
    rng = np.random.default_rng(random_state)
    
    # 전경색과 배경색을 각각 랜덤으로 할당
    fg = rng.integers(0, n_colors, size=n_samples)  # 전경색: 0-6
    bg = rng.integers(0, n_colors, size=n_samples)  # 배경색: 0-6

    # 전경색과 배경색이 같은 샘플 찾기
    same_mask = fg == bg
    
    # 전경색과 배경색이 같지 않을 때까지 반복
    while np.any(same_mask):
        # 같은 샘플의 배경색만 다시 랜덤 할당
        bg[same_mask] = rng.integers(0, n_colors, size=same_mask.sum())
        same_mask = fg == bg  # 다시 확인

    return fg.astype(np.int64), bg.astype(np.int64)


# ===== 전경색/배경색 라벨 할당 =====
# 모든 샘플(100,000개)에 대해 전경색과 배경색 라벨을 할당합니다.
y_fg_all, y_bg_all = assign_fg_bg_labels(
    n_samples=X_all.shape[0],
    n_colors=N_COLORS,  # 7 (ROYGBIV)
    random_state=777,
)

print("[INFO] fg labels shape:", y_fg_all.shape)
print("[INFO] bg labels shape:", y_bg_all.shape)
print("[INFO] any fg == bg?  :", np.any(y_fg_all == y_bg_all))  # False여야 함

# ===== EDA: 전경색/배경색 라벨 분포 확인 =====
# 전경색과 배경색의 분포가 균등한지 확인합니다.
fg_counts = pd.Series(y_fg_all).value_counts().sort_index()
bg_counts = pd.Series(y_bg_all).value_counts().sort_index()

print("\n[EDA] Foreground color label counts:")
print(fg_counts)
print("\n[EDA] Background color label counts:")
print(bg_counts)

# 전경색 분포를 막대 그래프로 시각화
plt.figure()
fg_counts.plot(kind="bar")
plt.title("Foreground Color Distribution (All 100k)")
plt.xlabel("fg color class")
plt.ylabel("Count")
plt.show()

# 배경색 분포를 막대 그래프로 시각화
plt.figure()
bg_counts.plot(kind="bar")
plt.title("Background Color Distribution (All 100k)")
plt.xlabel("bg color class")
plt.ylabel("Count")
plt.show()

# ===== 교차표: 전경색 vs 배경색 =====
# 전경색과 배경색의 조합 분포를 확인합니다.
# 대각선은 0 (fg != bg 보장)
fg_bg_ct = pd.crosstab(y_fg_all, y_bg_all)
print("\n[EDA] Crosstab (fg vs bg):")
print(fg_bg_ct)


def colorize_with_palette(X_gray: np.ndarray,
                          fg_labels: np.ndarray,
                          bg_labels: np.ndarray,
                          palette: np.ndarray):
    """
    그레이스케일 이미지를 팔레트를 사용하여 RGB 컬러 이미지로 변환하는 함수
    
    역할:
    - 그레이스케일 숫자 이미지를 전경색과 배경색을 사용하여 컬러 이미지로 변환
    - 픽셀 값(0-255)을 알파 값으로 사용하여 전경색과 배경색을 블렌딩
    - RGB 이미지와 평탄화된 특징 벡터를 모두 반환
    
    변환 원리:
    1. 배경색으로 전체 이미지 초기화
    2. 각 픽셀의 그레이스케일 값을 알파(alpha)로 사용
    3. alpha * 전경색 + (1-alpha) * 배경색으로 블렌딩
    4. 어두운 픽셀(숫자 부분) → 전경색에 가까움
    5. 밝은 픽셀(배경 부분) → 배경색에 가까움
    
    Parameters
    ----------
    X_gray : np.ndarray
        그레이스케일 이미지 배열 (N, 28, 28), float32
    fg_labels : np.ndarray
        전경색 라벨 배열 (N,), int64
    bg_labels : np.ndarray
        배경색 라벨 배열 (N,), int64
    palette : np.ndarray
        색상 팔레트 (7, 3), uint8
        각 행은 RGB 값 (예: [255, 0, 0] = 빨강)
    
    Returns
    -------
    X_rgb : np.ndarray
        RGB 컬러 이미지 배열 (N, 28, 28, 3), uint8
    X_flat : np.ndarray
        평탄화된 특징 벡터 (N, 2352), float32
        2352 = 28 * 28 * 3 (RGB 채널 포함)
    """
    N = X_gray.shape[0]
    # RGB 이미지를 저장할 배열 초기화 (N, 28, 28, 3)
    X_rgb = np.zeros((N, IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)
    # 그레이스케일 값을 [0, 1] 범위로 정규화 (블렌딩에 사용)
    X_norm = X_gray / 255.0

    # 각 샘플에 대해 컬러화 수행
    for i in range(N):
        # 전경색과 배경색을 팔레트에서 가져오기
        fg_color = palette[fg_labels[i]].astype(np.float32)  # (3,)
        bg_color = palette[bg_labels[i]].astype(np.float32)  # (3,)

        # 배경색으로 전체 이미지 초기화
        img_rgb = np.tile(bg_color[None, None, :], (IMG_HEIGHT, IMG_WIDTH, 1))
        
        # 그레이스케일 값을 알파로 사용 (어두운 부분 = 숫자 = 전경색)
        alpha = X_norm[i][..., None]  # (28, 28, 1)
        
        # 알파 블렌딩: alpha * 전경색 + (1-alpha) * 배경색
        img_rgb = alpha * fg_color + (1.0 - alpha) * img_rgb

        # [0, 255] 범위로 클리핑하고 uint8로 변환
        X_rgb[i] = img_rgb.clip(0, 255).astype(np.uint8)

    # RGB 이미지를 평탄화하여 특징 벡터 생성 (28*28*3 = 2352 차원)
    X_flat = X_rgb.reshape(N, -1).astype(np.float32)
    return X_rgb, X_flat


# ===== 컬러화 실행 =====
# 모든 그레이스케일 이미지를 전경색/배경색을 사용하여 RGB 컬러 이미지로 변환
X_rgb_all, X_flat_all = colorize_with_palette(
    X_gray=X_all,           # 그레이스케일 이미지 (100000, 28, 28)
    fg_labels=y_fg_all,    # 전경색 라벨 (100000,)
    bg_labels=y_bg_all,    # 배경색 라벨 (100000,)
    palette=COLOR_PALETTE,  # 색상 팔레트 (7, 3)
)

print("[INFO] Colored RGB shape:", X_rgb_all.shape)   # (100000, 28, 28, 3)
print("[INFO] Flattened shape  :", X_flat_all.shape)  # (100000, 2352)

# ===== 컬러화 결과 시각화 =====
# 컬러화가 올바르게 수행되었는지 확인하기 위한 시각화

def plot_colored_samples(X_rgb, y_digit, y_fg, y_bg, n=12):
    """
    컬러화된 샘플을 시각화하는 함수
    
    역할:
    - 컬러화된 샘플 중 n개를 랜덤 선택하여 표시
    - 각 샘플의 숫자, 전경색, 배경색 정보를 제목에 표시
    - 컬러화가 올바르게 수행되었는지 확인
    
    Parameters
    ----------
    X_rgb : np.ndarray
        RGB 컬러 이미지 배열 (N, 28, 28, 3)
    y_digit : np.ndarray
        숫자 라벨 배열 (N,)
    y_fg : np.ndarray
        전경색 라벨 배열 (N,)
    y_bg : np.ndarray
        배경색 라벨 배열 (N,)
    n : int
        표시할 샘플 수 (기본값: 12)
    """
    idx = np.random.choice(len(X_rgb), size=n, replace=False)
    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(cols * 2.2, rows * 2.2))
    for k, i in enumerate(idx):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(X_rgb[i])  # RGB 이미지 표시 (cmap 불필요)
        plt.title(f"d={y_digit[i]}, fg={y_fg[i]}, bg={y_bg[i]}")  # 정보 표시
        plt.axis("off")
    plt.suptitle("Colored MNIST Samples")
    plt.tight_layout()
    plt.show()

plot_colored_samples(X_rgb_all, y_all, y_fg_all, y_bg_all, n=12)

In [ ]:
# ======================================
# Cell 9. Train / Val split (80:20, no test)
# ======================================
# 이 셀은 전체 데이터셋을 학습용(train)과 검증용(val)으로 분할합니다.
# 비율: 80% 학습, 20% 검증 (테스트 세트는 별도로 분리하지 않음)
# 숫자 라벨을 기준으로 계층화(stratify)하여 각 클래스의 비율을 유지합니다.

# ===== Train/Val 분할 =====
# train_test_split을 사용하여 모든 데이터를 한 번에 분할
# - X_flat_all: 평탄화된 이미지 특징 (100000, 2352)
# - y_all: 숫자 라벨 (100000,)
# - y_fg_all: 전경색 라벨 (100000,)
# - y_bg_all: 배경색 라벨 (100000,)
# - source_all: 출처 정보 (100000,)
# 
# stratify=y_all: 숫자 라벨을 기준으로 계층화
#   → 각 숫자 클래스가 train/val에 동일한 비율로 분배됨
X_train, X_val, y_digit_train, y_digit_val, y_fg_train, y_fg_val, y_bg_train, y_bg_val, source_train, source_val = train_test_split(
    X_flat_all,      # 평탄화된 이미지 특징
    y_all,           # 숫자 라벨
    y_fg_all,        # 전경색 라벨
    y_bg_all,        # 배경색 라벨
    source_all,      # 출처 정보
    test_size=0.2,   # 검증 세트 비율: 20%
    random_state=RANDOM_SEED,  # 재현성 보장
    stratify=y_all,  # 숫자 라벨 기준 계층화
)

# ===== 분할 결과 확인 =====
print("[INFO] Final split shapes (train / val only):")
print("X_train       :", X_train.shape)        # (80000, 2352)
print("X_val         :", X_val.shape)          # (20000, 2352)
print("y_digit_train :", y_digit_train.shape)  # (80000,)
print("y_digit_val   :", y_digit_val.shape)    # (20000,)
print("y_fg_train    :", y_fg_train.shape)     # (80000,)
print("y_fg_val      :", y_fg_val.shape)       # (20000,)
print("y_bg_train    :", y_bg_train.shape)     # (80000,)
print("y_bg_val      :", y_bg_val.shape)       # (20000,)
print("source_train  :", source_train.shape)    # (80000,)
print("source_val    :", source_val.shape)      # (20000,)

def plot_split_distribution_2way(y_train, y_val, title_prefix="Digit"):
    """
    Train/Val 분할이 클래스 균형을 유지하는지 확인하는 함수
    
    역할:
    - Train과 Val 세트의 클래스 분포를 비교
    - 계층화(stratify)가 올바르게 작동했는지 확인
    - 각 클래스가 train/val에 비슷한 비율로 분배되었는지 시각화
    
    Parameters
    ----------
    y_train : np.ndarray
        학습 세트 라벨 배열
    y_val : np.ndarray
        검증 세트 라벨 배열
    title_prefix : str
        그래프 제목 접두사 (예: "Digit", "Foreground Color")
    """
    # 각 세트의 클래스별 개수 계산
    train_counts = pd.Series(y_train).value_counts().sort_index()
    val_counts   = pd.Series(y_val).value_counts().sort_index()

    print(f"\n[EDA] {title_prefix} distribution - train:")
    print(train_counts)
    print(f"[EDA] {title_prefix} distribution - val:")
    print(val_counts)

    # 모든 클래스 인덱스 수집
    idx = sorted(set(train_counts.index) | set(val_counts.index))
    # DataFrame으로 변환하여 비교 용이하게 함
    df = pd.DataFrame({
        "train": train_counts.reindex(idx),
        "val":   val_counts.reindex(idx),
    })

    # Train과 Val의 클래스 분포를 막대 그래프로 비교
    df.plot(kind="bar")
    plt.title(f"{title_prefix} Distribution per Split (train / val)")
    plt.xlabel(f"{title_prefix} class")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.show()

# ===== 숫자/전경색/배경색 클래스 균형 확인 =====
# 각 task의 클래스 분포가 train/val에서 균형을 유지하는지 확인
plot_split_distribution_2way(y_digit_train, y_digit_val, "Digit")
plot_split_distribution_2way(y_fg_train,    y_fg_val,    "Foreground Color")
plot_split_distribution_2way(y_bg_train,    y_bg_val,    "Background Color")

# ===== EDA: 출처 분포 확인 =====
# Train과 Val 세트에서 각 증강 방법(raw/deskew/font/geom)의 분포 확인
print("\n[EDA] Source distribution - train:")
print(pd.Series(source_train).value_counts())

print("\n[EDA] Source distribution - val:")
print(pd.Series(source_val).value_counts())

In [ ]:
# ======================================
# Cell 10. Save final NPZ (train/val only)
# ======================================
# 이 셀은 전처리 완료된 Colored MNIST 데이터셋을 NPZ 형식으로 저장합니다.
# 저장된 파일은 02_train_classical_ml.ipynb와 03_evaluation_and_plots.ipynb에서 사용됩니다.

# ===== NPZ 파일로 저장 =====
# np.savez_compressed: 여러 배열을 하나의 압축된 NPZ 파일에 저장
# 압축을 사용하여 파일 크기를 줄입니다.

np.savez_compressed(
    SAVE_PATH,  # 저장 경로 (configs/paths.yaml에서 로드)
    
    # ===== 특징 데이터 =====
    # 평탄화된 RGB 이미지 특징 벡터 (28*28*3 = 2352 차원)
    X_train=X_train,  # 학습용 이미지 (80000, 2352)
    X_val=X_val,      # 검증용 이미지 (20000, 2352)

    # ===== 라벨 데이터 =====
    # 3가지 task의 라벨: 숫자 분류, 전경색 분류, 배경색 분류
    y_digit_train=y_digit_train,  # 학습용 숫자 라벨 (80000,)
    y_digit_val=y_digit_val,      # 검증용 숫자 라벨 (20000,)
    y_fg_train=y_fg_train,        # 학습용 전경색 라벨 (80000,)
    y_fg_val=y_fg_val,            # 검증용 전경색 라벨 (20000,)
    y_bg_train=y_bg_train,        # 학습용 배경색 라벨 (80000,)
    y_bg_val=y_bg_val,            # 검증용 배경색 라벨 (20000,)

    # ===== 출처 정보 =====
    # 각 샘플이 어떤 증강 방법으로 생성되었는지 기록
    # "raw", "deskew", "font", "geom" 중 하나
    source_train=source_train,  # 학습용 출처 정보 (80000,)
    source_val=source_val,      # 검증용 출처 정보 (20000,)

    # ===== 메타데이터 =====
    # 데이터셋 생성에 사용된 설정값들을 저장하여 재현성 보장
    color_palette=COLOR_PALETTE,      # 색상 팔레트 (7, 3)
    random_seed=RANDOM_SEED,          # 랜덤 시드 (42)
    deskew_aug_target=DESKEW_AUG_TARGET,  # Deskew 증강 목표 수 (20000)
    font_aug_target=FONT_AUG_TARGET,      # Font 증강 목표 수 (10000)
    geom_aug_target=GEOM_AUG_TARGET,      # Geometric 증강 목표 수 (10000)
)

print(f"[INFO] Saved processed colored MNIST (train/val only) to: {SAVE_PATH}")

## # PART 2: 모델 학습

02_train_classical_ml.ipynb에서 가져온 코드입니다.

In [ ]:
# This cell is only needed if xgboost is not installed in your environment.
# In Colab, run this once and then restart the runtime if necessary.
# option Cell !!! 
# xgboost 안깔려 있으면 주석 풀고 실행!
#!pip install -q xgboost

In [ ]:
# ================================
# Cell 1.5. Config loader utilities
# ================================
# 이 셀은 Config 파일을 로드하는 유틸리티 함수들을 정의합니다.
# Config 파일을 통해 경로, 모델 하이퍼파라미터, 실험 설정을 중앙에서 관리합니다.

import yaml  # YAML 파일 파싱 라이브러리

def load_config(config_path):
    """
    YAML config 파일을 로드하여 dict 반환
    
    역할:
    - configs/ 디렉토리의 YAML 파일을 읽어서 Python 딕셔너리로 변환
    - 노트북 실행 위치에 관계없이 올바른 경로를 찾아서 config 파일 로드
    
    사용 예시:
        config = load_config("configs/paths.yaml")
        # → {'paths': {'data': {...}, 'results': {...}}}
    
    Parameters
    ----------
    config_path : str
        Config 파일 경로 (상대 또는 절대 경로)
        예: "configs/paths.yaml", "configs/models.yaml"
    
    Returns
    -------
    dict
        로드된 config 딕셔너리 (YAML 구조 그대로)
    """
    # 현재 작업 디렉토리 확인
    notebook_dir = Path.cwd()
    
    # 노트북이 notebooks/ 디렉토리 안에서 실행되는지 확인
    # → 프로젝트 루트를 자동으로 찾기 위함
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent  # notebooks/의 부모 = 프로젝트 루트
    else:
        project_root = notebook_dir  # 이미 루트에서 실행 중
    
    # Config 파일 경로 처리
    config_file = Path(config_path)
    if not config_file.is_absolute():  # 상대 경로인 경우
        config_file = project_root / config_path  # 프로젝트 루트 기준으로 변환
    
    # YAML 파일 읽기 및 파싱
    with open(config_file, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)  # 안전하게 YAML을 딕셔너리로 변환


def get_paths(config_path="configs/paths.yaml"):
    """
    경로 config를 로드하고 절대 경로로 변환
    
    역할:
    - paths.yaml에서 경로 정보를 읽어서 절대 경로 객체(Path)로 변환
    - 모든 노트북에서 일관된 경로를 사용할 수 있도록 함
    
    사용 예시:
        paths = get_paths()
        npz_path = paths["data"]["processed"]
        # → Path('/Users/.../data/processed/colored_mnist/colored_mnist_100k_train_val.npz')
    
    Parameters
    ----------
    config_path : str
        경로 config 파일 경로 (기본값: "configs/paths.yaml")
    
    Returns
    -------
    dict
        절대 경로로 변환된 경로 딕셔너리
        구조: {
            "data": {
                "raw_mnist": Path(...),
                "processed": Path(...),
                ...
            },
            "results": {
                "metrics": Path(...),
                "figures": Path(...),
                ...
            }
        }
    """
    # Config 파일 로드
    config = load_config(config_path)
    paths = config['paths']  # 'paths' 키의 값 추출
    
    # 프로젝트 루트 찾기
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    # 상대 경로를 절대 경로(Path 객체)로 변환
    resolved = {}
    for key, value in paths.items():
        if isinstance(value, dict):  # 중첩된 딕셔너리인 경우 (예: data, results)
            # 각 하위 경로도 절대 경로로 변환 (None 값은 그대로 유지)
            resolved[key] = {
                k: project_root / v if v is not None else None 
                for k, v in value.items()
            }
        else:  # 단일 경로인 경우
            resolved[key] = project_root / value if value is not None else None
    
    return resolved


def load_model_config(config_path="configs/models.yaml"):
    """
    모델 config를 로드
    
    역할:
    - models.yaml에서 모델별 하이퍼파라미터를 읽어옴
    - BASE_MODELS, PARAM_GRIDS, NEEDS_SCALING 등을 동적으로 생성하는데 사용
    
    사용 예시:
        model_config = load_model_config()
        knn_params = model_config["models"]["knn"]["base"]
        # → {'n_neighbors': 5, 'p': 2, 'metric': 'minkowski', ...}
    
    Parameters
    ----------
    config_path : str
        모델 config 파일 경로 (기본값: "configs/models.yaml")
    
    Returns
    -------
    dict
        모델 설정 딕셔너리
        구조: {
            "models": {
                "knn": {"base": {...}, "grid_search": {...}, "needs_scaling": true},
                "xgb": {...},
                ...
            },
            "use_gridsearch": {...}
        }
    """
    return load_config(config_path)


def load_experiment_config(config_path):
    """
    실험 config를 로드
    
    역할:
    - experiments/ 디렉토리의 실험별 설정 파일을 읽어옴
    - 어떤 모델을 사용할지, GridSearch를 사용할지 등의 실험 설정 관리
    
    사용 예시:
        exp_config = load_experiment_config("configs/experiments/digit_best.yaml")
        active_models = exp_config["active_models"]
        # → ['knn', 'svm', 'tree', 'rf', 'xgb']
    
    Parameters
    ----------
    config_path : str
        실험 config 파일 경로
        예: "configs/experiments/digit_best.yaml"
    
    Returns
    -------
    dict
        실험 설정 딕셔너리
        구조: {
            "experiment": {"name": "...", "task": "digit", ...},
            "active_models": [...],
            "use_gridsearch": {...},
            "evaluation": {...}
        }
    """
    return load_config(config_path)


print("[INFO] Config loader utilities defined.")



In [ ]:
# ================================
# Cell 1. Imports & global config
# ================================
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

# 로지스틱 회귀 제거 (중간보고서 기준)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
# 앙상블 제거 (중간보고서 기준)

# PCA for KNN Digit Task (보고서: 97.93% 달성)
from sklearn.decomposition import PCA

# XGBoost (install if necessary: !pip install xgboost)
from xgboost import XGBClassifier

# Model persistence
import joblib

# For notebook
%matplotlib inline

# Global random seed for reproducibility
RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Matplotlib configuration
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

print("[INFO] Libraries imported.")

In [ ]:
# ===================================
# Cell 2. Paths & dataset information (Config-based)
# ===================================
# 이 셀은 데이터 경로와 결과 저장 경로를 설정합니다.
# 모든 경로는 configs/paths.yaml에서 중앙 관리되므로, 경로 변경 시 YAML만 수정하면 됩니다.

# ===== Config에서 경로 로드 =====
# get_paths() 함수를 사용하여 configs/paths.yaml의 모든 경로를 딕셔너리로 가져옵니다.
paths = get_paths("configs/paths.yaml")
# paths 구조:
# {
#   "data": {
#     "raw_mnist": Path(...),
#     "processed": Path(...)
#   },
#   "results": {
#     "metrics": Path(...),
#     "figures": Path(...),
#     "models": Path(...)
#   }
# }

# ===== 입력 데이터 경로 =====
# 01_preprocessing_colored_mnist.ipynb에서 생성한 전처리된 데이터셋 경로
# 이 파일에는 train/val split된 Colored MNIST 데이터가 저장되어 있습니다.
NPZ_PATH = paths["data"]["processed"]
# 예: data/processed/colored_mnist/colored_mnist_100k_train_val.npz
# 구조:
#   - X_train: (80000, 2352) - 학습용 이미지 (28*28*3 = 2352차원으로 평탄화)
#   - X_val: (20000, 2352) - 검증용 이미지
#   - y_digit_train/val: (80000,), (20000,) - 숫자 라벨 (0-9)
#   - y_fg_train/val: (80000,), (20000,) - 전경색 라벨 (0-6: ROYGBIV)
#   - y_bg_train/val: (80000,), (20000,) - 배경색 라벨 (0-6: ROYGBIV)

# ===== 프로젝트 루트 디렉토리 =====
# 현재 작업 디렉토리를 확인하고, 프로젝트 루트를 찾습니다.
BASE_DIR = Path.cwd()
if 'notebooks' in str(BASE_DIR):
    BASE_DIR = BASE_DIR.parent  # notebooks/의 부모 = 프로젝트 루트

# ===== 결과 저장 경로 =====
# 모델 학습 결과(메트릭, 시각화)를 저장할 디렉토리들
RESULTS_METRICS_DIR = paths["results"]["metrics"]  # CSV 형식의 평가 지표 저장 경로
RESULTS_FIGURES_DIR = paths["results"]["figures"]   # PNG 형식의 시각화 결과 저장 경로
# 예: results/metrics/, results/figures/

# 디렉토리가 없으면 자동으로 생성
os.makedirs(RESULTS_METRICS_DIR, exist_ok=True)
os.makedirs(RESULTS_FIGURES_DIR, exist_ok=True)

# ===== 이미지 크기 상수 =====
# Colored MNIST 이미지의 크기 (높이, 너비, 채널)
IMG_SHAPE = (28, 28, 3)  # 28x28 픽셀, RGB 3채널
# 평탄화하면 28*28*3 = 2352 차원의 벡터가 됩니다.

# ===== 정보 출력 =====
# 설정된 경로를 확인하기 위한 출력
print("[INFO] BASE_DIR      :", BASE_DIR)
print("[INFO] NPZ_PATH      :", NPZ_PATH)
print("[INFO] METRICS_DIR   :", RESULTS_METRICS_DIR)
print("[INFO] FIGURES_DIR   :", RESULTS_FIGURES_DIR)

print("[INFO] BASE_DIR      :", BASE_DIR)
print("[INFO] NPZ_PATH      :", NPZ_PATH)
print("[INFO] METRICS_DIR   :", RESULTS_METRICS_DIR)
print("[INFO] FIGURES_DIR   :", RESULTS_FIGURES_DIR)

In [ ]:
# ==========================================
# Cell 3. Load processed Colored MNIST splits
# ==========================================
# 이 셀은 01_preprocessing_colored_mnist.ipynb에서 생성한 전처리된 데이터를 로드합니다.
# 데이터는 train/val로 이미 분할되어 있으며, 3가지 task(digit, fg, bg)의 라벨이 모두 포함되어 있습니다.

# ===== 데이터 파일 존재 확인 =====
# NPZ 파일이 존재하는지 확인하고, 없으면 에러를 발생시킵니다.
# 이 파일은 01 노트북을 먼저 실행해야 생성됩니다.
if not os.path.exists(NPZ_PATH):
    raise FileNotFoundError(f"[ERROR] Processed dataset not found at: {NPZ_PATH}")

# ===== NPZ 파일 로드 =====
# NumPy의 npz 형식으로 저장된 데이터를 로드합니다.
# npz는 여러 배열을 하나의 파일에 저장하는 압축 형식입니다.
data = np.load(NPZ_PATH)

# ===== 이미지 데이터 로드 =====
# 평탄화된 컬러 이미지: (N, 2352) 형태
# - 28*28*3 = 2352 차원 (원본 28x28 RGB 이미지를 1차원 벡터로 변환)
# - float32 타입, 픽셀 값 범위: [0, 255]
X_train = data["X_train"]  # 학습용 이미지: (80000, 2352)
X_val   = data["X_val"]    # 검증용 이미지: (20000, 2352)

# ===== Task 1: Digit Classification 라벨 =====
# 숫자 분류 task의 라벨 (0-9, 총 10개 클래스)
y_digit_train = data["y_digit_train"]  # 학습용 숫자 라벨: (80000,)
y_digit_val   = data["y_digit_val"]    # 검증용 숫자 라벨: (20000,)

# ===== Task 2: Foreground Color Classification 라벨 =====
# 전경색(숫자 색상) 분류 task의 라벨 (0-6: ROYGBIV, 총 7개 클래스)
# 0=Red, 1=Orange, 2=Yellow, 3=Green, 4=Blue, 5=Indigo, 6=Violet
y_fg_train = data["y_fg_train"]  # 학습용 전경색 라벨: (80000,)
y_fg_val   = data["y_fg_val"]    # 검증용 전경색 라벨: (20000,)

# ===== Task 3: Background Color Classification 라벨 =====
# 배경색 분류 task의 라벨 (0-6: ROYGBIV, 총 7개 클래스)
y_bg_train = data["y_bg_train"]  # 학습용 배경색 라벨: (80000,)
y_bg_val   = data["y_bg_val"]    # 검증용 배경색 라벨: (20000,)

# ===== 데이터 출처 정보 (선택적) =====
# 각 샘플이 어떤 증강 방법으로 생성되었는지 기록
# "raw": 원본 MNIST
# "deskew": Deskew 증강
# "font": Font 증강
# "geom": Geometric 증강
source_train = data["source_train"]  # 학습 데이터 출처: (80000,)
source_val   = data["source_val"]    # 검증 데이터 출처: (20000,)

# ===== 데이터 로드 정보 출력 =====
print("[INFO] Dataset loaded from npz.")
print("  X_train shape:", X_train.shape, "dtype:", X_train.dtype)
print("  X_val   shape:", X_val.shape,   "dtype:", X_val.dtype)
print("  y_digit_train shape:", y_digit_train.shape)
print("  y_fg_train    shape:", y_fg_train.shape)
print("  y_bg_train    shape:", y_bg_train.shape)

# ===== 데이터 일관성 검증 =====
# 데이터가 올바르게 로드되었는지 확인하는 검증 단계
# 1. 이미지 차원이 28*28*3 = 2352인지 확인
assert X_train.shape[1] == 28 * 28 * 3, "[ERROR] Feature dimension must be 28*28*3."
# 2. 학습 데이터의 샘플 수가 모든 라벨과 일치하는지 확인
assert len(X_train) == len(y_digit_train) == len(y_fg_train) == len(y_bg_train)
# 3. 검증 데이터의 샘플 수가 모든 라벨과 일치하는지 확인
assert len(X_val)   == len(y_digit_val)   == len(y_fg_val)   == len(y_bg_val)

print("[INFO] Basic consistency checks passed.")


In [ ]:
# =====================================================
# Cell 4. Task selection helper (digit / fg / bg labels)
# =====================================================
# Map each task name to its corresponding labels and description
TASK_LABEL_INFO = {
    "digit": {
        "y_train": y_digit_train,
        "y_val":   y_digit_val,
        "task_desc": "Digit classification (0-9)",
    },
    "fg": {
        "y_train": y_fg_train,
        "y_val":   y_fg_val,
        "task_desc": "Foreground color classification (7 classes, ROYG BIV)",
    },
    "bg": {
        "y_train": y_bg_train,
        "y_val":   y_bg_val,
        "task_desc": "Background color classification (7 classes, ROYG BIV)",
    },
}

# Which tasks to run (you can restrict this list when debugging)
ACTIVE_TASKS = ["digit", "fg", "bg"]  # 모든 task 실행
print("[INFO] Active tasks:", ACTIVE_TASKS)


In [ ]:
# ==========================================
# Cell 5. Visualization helpers (EDA)
# ==========================================
def reconstruct_images(X_flat, n_samples=16):
    """
    Reconstruct RGB images from flattened feature vectors for visualization.
    """
    n = min(n_samples, X_flat.shape[0])
    idxs = np.random.choice(X_flat.shape[0], size=n, replace=False)
    imgs = X_flat[idxs].reshape(n, *IMG_SHAPE)
    return imgs, idxs


def plot_sample_images(X_flat, y, title, n_samples=16):
    """
    Plot a grid of sample images with their labels.
    """
    imgs, idxs = reconstruct_images(X_flat, n_samples=n_samples)
    n = imgs.shape[0]
    cols = min(8, n)
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(cols * 1.5, rows * 1.5))
    for i, (img, idx) in enumerate(zip(imgs, idxs)):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img / 255.0)  # scale to [0,1] for visualization
        ax.axis("off")
        ax.set_title(str(int(y[idx])), fontsize=8)
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_label_distribution_2way(y_train, y_val, task_name):
    """
    Plot label distribution for train/val splits for a given task.
    """
    def counts(y):
        return pd.Series(y).value_counts().sort_index()

    train_c = counts(y_train)
    val_c   = counts(y_val)

    df = pd.DataFrame({
        "label": train_c.index,
        "train": train_c.values,
        "val":   val_c.values,
    })

    print(f"\n[EDA] Label distribution for task = {task_name}")
    print(df)

    x = np.arange(len(df["label"]))
    width = 0.35

    plt.figure(figsize=(8, 4))
    plt.bar(x - width / 2, df["train"], width=width, label="train")
    plt.bar(x + width / 2, df["val"],   width=width, label="val")

    plt.xticks(x, df["label"])
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.title(f"Label distribution per split ({task_name})")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_source_distribution(source_train, source_val):
    """
    Plot distribution of data sources (raw / deskew / font / geom) in train/val.
    """
    train_counts = pd.Series(source_train).value_counts().sort_index()
    val_counts   = pd.Series(source_val).value_counts().sort_index()

    df = pd.DataFrame({
        "source": train_counts.index,
        "train": train_counts.values,
        "val":   val_counts.reindex(train_counts.index).values,
    })

    print("\n[EDA] Source distribution (train/val):")
    print(df)

    x = np.arange(len(df["source"]))
    width = 0.35

    plt.figure(figsize=(8, 4))
    plt.bar(x - width / 2, df["train"], width=width, label="train")
    plt.bar(x + width / 2, df["val"],   width=width, label="val")
    plt.xticks(x, df["source"])
    plt.xlabel("Source")
    plt.ylabel("Count")
    plt.title("Source distribution per split (raw / deskew / font / geom)")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ==========================================
# Cell 6. Evaluation & plotting utilities
# ==========================================
def compute_metrics(y_true, y_pred, average="macro"):
    """
    Compute accuracy, precision, recall, and F1-score (macro-averaged by default).
    """
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return acc, prec, rec, f1


def plot_confusion_matrix(y_true, y_pred, classes, title, save_path=None):
    """
    Plot confusion matrix as a heatmap.
    """
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes,
        cmap="YlGnBu",
    )
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)
        print("[INFO] Saved confusion matrix to:", save_path)

    plt.show()


def show_prediction_examples(model_name, pipeline, X, y_true, task_name, n=12):
    """
    Show images with true vs predicted labels for qualitative verification.
    Uses validation set here (since no test set exists).
    """
    n = min(n, X.shape[0])
    idxs = np.random.choice(X.shape[0], size=n, replace=False)
    X_sample = X[idxs]
    y_sample_true = y_true[idxs]
    y_sample_pred = pipeline.predict(X_sample)

    imgs = X_sample.reshape(n, *IMG_SHAPE)

    cols = min(6, n)
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(cols * 2, rows * 2.2))
    for i in range(n):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(imgs[i] / 255.0)
        ax.axis("off")
        ax.set_title(
            f"T:{int(y_sample_true[i])} / P:{int(y_sample_pred[i])}",
            fontsize=8,
        )
    plt.suptitle(f"{task_name} - {model_name}: prediction examples (val set)", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_feature_importance_heatmap(feature_importances, title, save_path=None):
    """
    Visualize feature importances (flattened 2352-dim vector) as a heatmap over 28x28 pixels,
    by averaging the 3 RGB channels.
    """
    if feature_importances.shape[0] != 28 * 28 * 3:
        print("[WARN] Unexpected feature_importances length:", feature_importances.shape[0])
        return

    # Reshape to (28, 28, 3) and average across RGB channels
    imp_3d = feature_importances.reshape(28, 28, 3)
    imp_2d = imp_3d.mean(axis=2)

    plt.figure(figsize=(4, 4))
    sns.heatmap(imp_2d, cmap="viridis")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)
        print("[INFO] Saved feature importance heatmap to:", save_path)

    plt.show()

In [ ]:
# ==========================================
# Cell 7. Model definitions & GridSearch configs (Config-based)
# ==========================================
# 이 셀은 configs/models.yaml에서 모델 설정을 로드하여 모델 인스턴스를 동적으로 생성합니다.
# 하이퍼파라미터를 코드에 하드코딩하지 않고, YAML 파일에서 관리할 수 있습니다.

# ===== Config 파일에서 모델 설정 로드 =====
# models.yaml 파일을 읽어서 모든 모델의 하이퍼파라미터를 가져옵니다.
model_config = load_model_config("configs/models.yaml")
models_cfg = model_config["models"]  # 모델별 설정 딕셔너리
use_gridsearch_cfg = model_config["use_gridsearch"]  # GridSearch 사용 여부 설정

# ===== BASE_MODELS 딕셔너리 생성 (Task별 하이퍼파라미터 지원) =====
# 각 모델의 Task별 하이퍼파라미터를 사용하여 모델 인스턴스를 생성합니다.
# 이 딕셔너리는 train_single_model() 함수에서 사용됩니다.
# 구조: BASE_MODELS[task_name][model_name] = model_instance

def create_model_instance(model_name, task_name, model_params):
    """
    Task별 하이퍼파라미터를 사용하여 모델 인스턴스를 생성합니다.
    
    Parameters
    ----------
    model_name : str
        모델 이름 (knn, svm, tree, rf, xgb)
    task_name : str
        Task 이름 (digit, fg, bg)
    model_params : dict
        models.yaml에서 로드한 모델 설정
    
    Returns
    -------
    model instance
        생성된 모델 인스턴스
    """
    # Task별 하이퍼파라미터 가져오기
    if task_name in model_params:
        task_params = model_params[task_name].copy()
    else:
        # Task별 설정이 없으면 기본값 사용 (하위 호환성)
        task_params = model_params.get("base", {}).copy()
    
    # Pipeline 관련 파라미터 제거 (모델 파라미터가 아님)
    task_params.pop("use_pca", None)
    task_params.pop("pca_n_components", None)
    
    # 재현성을 위해 random_state가 필요한 모델에 시드 추가
    if model_name in ["tree", "rf", "xgb"]:
        task_params["random_state"] = RANDOM_SEED
    
    # 모델 타입에 따라 적절한 클래스로 인스턴스 생성
    if model_name == "knn":
        return KNeighborsClassifier(**task_params)
    elif model_name == "svm":
        return SVC(**task_params)
    elif model_name == "tree":
        return DecisionTreeClassifier(**task_params)
    elif model_name == "rf":
        return RandomForestClassifier(**task_params)
    elif model_name == "xgb":
        return XGBClassifier(**task_params)
    else:
        raise ValueError(f"Unknown model: {model_name}")

# BASE_MODELS를 Task별로 구성 (나중에 train_single_model에서 task_name으로 접근)
# 실제로는 train_single_model에서 동적으로 생성하므로, 여기서는 구조만 정의
BASE_MODELS = {}  # 사용하지 않음, train_single_model에서 동적 생성

# ===== NEEDS_SCALING 딕셔너리 생성 =====
# 각 모델이 StandardScaler를 필요로 하는지 여부를 저장합니다.
# - True: StandardScaler 필요 (KNN, SVM, Logistic Regression)
# - False: StandardScaler 불필요 (Tree 계열 모델들)
# build_pipeline() 함수에서 이 정보를 사용하여 Pipeline을 구성합니다.
NEEDS_SCALING = {
    model_name: model_params["needs_scaling"]
    for model_name, model_params in models_cfg.items()
}

# ===== PARAM_GRIDS 딕셔너리 생성 =====
# GridSearchCV에 사용할 하이퍼파라미터 그리드를 생성합니다.
# 각 모델의 "grid_search" 섹션에 정의된 파라미터 범위를 사용합니다.
PARAM_GRIDS = {}
for model_name, model_params in models_cfg.items():
    if "grid_search" in model_params:  # GridSearch 설정이 있는 경우만
        PARAM_GRIDS[model_name] = model_params["grid_search"]
# 예: PARAM_GRIDS["knn"] = {"clf__n_neighbors": [3, 5, 7, 11], ...}

# ===== USE_GRIDSEARCH 딕셔너리 생성 =====
# 각 모델에 대해 GridSearchCV를 사용할지 여부를 설정합니다.
# False로 설정하면 기본 하이퍼파라미터만 사용합니다.
USE_GRIDSEARCH = use_gridsearch_cfg.copy()

# ===== 실행할 모델 선택 =====
# 실제로 학습할 모델 목록을 지정합니다.

ACTIVE_MODELS = ["knn", "svm", "tree", "rf", "xgb"]  # 로지스틱 회귀 제거
print("[INFO] Active models:", ACTIVE_MODELS)
print("[INFO] Models config loaded from:", list(models_cfg.keys()))


def build_pipeline(model_name, base_estimator, task_name=None, model_params=None):
    """
    Build sklearn Pipeline for a given model:
      - If NEEDS_SCALING[model_name] is True:
          [StandardScaler] -> [Classifier]
      - KNN + Digit Task + use_pca=True:
          [StandardScaler] -> [PCA] -> [Classifier]
      - Otherwise:
          [Classifier] only
    """
    steps = []
    
    # StandardScaler 추가 (필요한 모델만)
    if NEEDS_SCALING[model_name]:
        steps.append(("scaler", StandardScaler()))
    
    # PCA 추가 (KNN + Digit Task에서만 적용)
    if model_name == "knn" and task_name is not None and model_params is not None:
        task_cfg = model_params.get(task_name, {})
        use_pca = task_cfg.get("use_pca", False)
        if use_pca:
            pca_n_components = task_cfg.get("pca_n_components", 0.99)
            print(f"[INFO] PCA 적용: n_components={pca_n_components}")
            steps.append(("pca", PCA(n_components=pca_n_components, random_state=RANDOM_SEED)))
    
    # Classifier 추가
    steps.append(("clf", base_estimator))
    
    pipe = Pipeline(steps)
    return pipe


def train_single_model(
    model_name,
    X_train,
    y_train,
    X_val,
    y_val,
    task_name,
    use_gridsearch=True,
):
    """
    Train a single model (optionally with GridSearchCV) and evaluate on val set.
    Task별 하이퍼파라미터를 사용하여 모델을 생성합니다.

    Returns
    -------
    best_pipeline : trained Pipeline
    metrics_val   : dict with accuracy, precision, recall, f1 (macro)
    metrics_train : dict with train metrics
    gap           : train_acc - val_acc
    """
    print("\n==============================")
    print(f"[TASK: {task_name}] Training model: {model_name}")
    print("==============================")

    # Task별 하이퍼파라미터로 모델 인스턴스 생성
    model_params = models_cfg[model_name]
    base_estimator = create_model_instance(model_name, task_name, model_params)
    
    # Pipeline 생성 (KNN+Digit의 경우 PCA 포함)
    pipe = build_pipeline(model_name, base_estimator, task_name, model_params)

    if use_gridsearch and model_name in PARAM_GRIDS:
        param_grid = PARAM_GRIDS[model_name]
        print("[INFO] Running GridSearchCV for", model_name)
        grid = GridSearchCV(
            pipe,
            param_grid=param_grid,
            cv=3,
            n_jobs=-1,
            scoring="accuracy",
            verbose=1,
        )
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        print("[INFO] Best params:", grid.best_params_)
        print("[INFO] Best CV accuracy:", grid.best_score_)
    else:
        print("[INFO] Training without GridSearch (fixed hyperparameters).")
        pipe.fit(X_train, y_train)
        best_pipeline = pipe

    # Validation performance
    y_val_pred = best_pipeline.predict(X_val)
    val_acc, val_prec, val_rec, val_f1 = compute_metrics(y_val, y_val_pred, average="macro")

    print("\n[VAL] metrics (macro) for", model_name)
    print(f"  accuracy : {val_acc:.4f}")
    print(f"  precision: {val_prec:.4f}")
    print(f"  recall   : {val_rec:.4f}")
    print(f"  f1-score : {val_f1:.4f}")

    print("\n[VAL] classification_report:")
    print(classification_report(y_val, y_val_pred, digits=4))

    # Train performance (과대적합 진단용)
    y_train_pred = best_pipeline.predict(X_train)
    train_acc, train_prec, train_rec, train_f1 = compute_metrics(y_train, y_train_pred, average="macro")

    print("\n[TRAIN] metrics (macro) for", model_name)
    print(f"  accuracy : {train_acc:.4f}")
    print(f"  precision: {train_prec:.4f}")
    print(f"  recall   : {train_rec:.4f}")
    print(f"  f1-score : {train_f1:.4f}")

    # Gap 계산 (과대적합 진단)
    gap = train_acc - val_acc
    print(f"\n[GAP] Train - Val: {gap:.4f}")
    
    # 과대적합/과소적합 경고
    if gap > 0.05:
        print(f"[WARNING] 과대적합 가능성! (Gap: {gap:.4f})")
    elif gap < 0.01:
        print(f"[INFO] 적절한 적합 (Gap: {gap:.4f})")
    else:
        print(f"[INFO] 약간의 과대적합 (Gap: {gap:.4f})")

    metrics_val = {
        "accuracy": val_acc,
        "precision": val_prec,
        "recall": val_rec,
        "f1": val_f1,
    }
    
    metrics_train = {
        "accuracy": train_acc,
        "precision": train_prec,
        "recall": train_rec,
        "f1": train_f1,
    }

    return best_pipeline, metrics_val, metrics_train, gap



In [ ]:
# ==========================================
# Cell 8. Main training loop over tasks/models
# ==========================================
all_results = []  # will store metrics for all (task, model, split)

# EDA: source distribution (once)
plot_source_distribution(source_train, source_val)

for task_name in ACTIVE_TASKS:
    info = TASK_LABEL_INFO[task_name]
    y_train_task = info["y_train"]
    y_val_task   = info["y_val"]
    task_desc    = info["task_desc"]

    print("\n\n##########################################")
    print(f"### Task: {task_name} - {task_desc}")
    print("##########################################")

    # EDA: label distribution
    plot_label_distribution_2way(y_train_task, y_val_task, task_name)

    # EDA: sample images (using training split, labels from this task)
    plot_sample_images(
        X_flat=X_train,
        y=y_train_task,
        title=f"Sample images (train) - labels: {task_name}",
        n_samples=16,
    )

    # Classes sorted for confusion matrix axis
    classes = np.sort(np.unique(y_train_task))

    # Dictionary to store trained pipelines for this task
    trained_pipelines = {}

    # Loop over models selected in ACTIVE_MODELS
    for model_name in ACTIVE_MODELS:
        # Train (with or without GridSearch depending on USE_GRIDSEARCH flag)
        use_grid = USE_GRIDSEARCH.get(model_name, False)
        best_pipe, val_metrics, train_metrics, gap = train_single_model(
            model_name=model_name,
            X_train=X_train,
            y_train=y_train_task,
            X_val=X_val,
            y_val=y_val_task,
            task_name=task_name,
            use_gridsearch=use_grid,
        )

        trained_pipelines[model_name] = best_pipe

        # 모델 저장
        RESULTS_MODELS_DIR = paths["results"]["models"]
        os.makedirs(RESULTS_MODELS_DIR, exist_ok=True)
        model_path = os.path.join(RESULTS_MODELS_DIR, f"{task_name}_{model_name}.joblib")
        joblib.dump(best_pipe, model_path)
        print(f"[INFO] Model saved to: {model_path}")

        # Evaluate on validation set (이미 train_single_model에서 계산됨, 중복 제거)
        acc_val = val_metrics["accuracy"]
        prec_val = val_metrics["precision"]
        rec_val = val_metrics["recall"]
        f1_val = val_metrics["f1"]
        
        train_acc = train_metrics["accuracy"]
        train_prec = train_metrics["precision"]
        train_rec = train_metrics["recall"]
        train_f1 = train_metrics["f1"]

        print(f"\n[VAL FINAL] {task_name} - {model_name}")
        print(f"  accuracy : {acc_val:.4f}")
        print(f"  precision: {prec_val:.4f}")
        print(f"  recall   : {rec_val:.4f}")
        print(f"  f1-score : {f1_val:.4f}")

        # Confusion matrix on validation set (y_val_pred 재계산)
        y_val_pred = best_pipe.predict(X_val)
        cm_title = f"{task_name} - {model_name} (val set)"
        cm_filename = f"cm_{task_name}_{model_name}_val.png"
        cm_path = os.path.join(RESULTS_FIGURES_DIR, cm_filename)
        plot_confusion_matrix(
            y_true=y_val_task,
            y_pred=y_val_pred,
            classes=classes,
            title=cm_title,
            save_path=cm_path,
        )

        # Feature importance for tree-based models (DecisionTree, RandomForest, XGBoost)
        if model_name in ["tree", "rf", "xgb"]:
            clf = best_pipe.named_steps["clf"]
            if hasattr(clf, "feature_importances_"):
                fi = clf.feature_importances_
                fi_title = f"{task_name} - {model_name} feature importance (avg over RGB)"
                fi_filename = f"feat_importance_{task_name}_{model_name}.png"
                fi_path = os.path.join(RESULTS_FIGURES_DIR, fi_filename)
                plot_feature_importance_heatmap(
                    feature_importances=fi,
                    title=fi_title,
                    save_path=fi_path,
                )
            else:
                print(f"[WARN] Model {model_name} does not expose feature_importances_.")

        # Qualitative prediction examples (true vs predicted labels) on val set
        show_prediction_examples(
            model_name=model_name,
            pipeline=best_pipe,
            X=X_val,
            y_true=y_val_task,
            task_name=task_name,
            n=12,
        )

        # Store train metrics
        all_results.append({
            "task": task_name,
            "task_desc": task_desc,
            "model": model_name,
            "split": "train",
            "accuracy": train_acc,
            "precision": train_prec,
            "recall": train_rec,
            "f1": train_f1,
        })
        
        # Store validation metrics
        all_results.append({
            "task": task_name,
            "task_desc": task_desc,
            "model": model_name,
            "split": "val",
            "accuracy": acc_val,
            "precision": prec_val,
            "recall": rec_val,
            "f1": f1_val,
            "gap": gap,  # Train-Val gap for overfitting diagnosis
        })



In [ ]:
# ==========================================
# Cell 9. Metrics table (for report) + save CSV
# ==========================================
results_df = pd.DataFrame(all_results)

# Sort by task, split, and accuracy (descending on accuracy)
results_df_sorted = results_df.sort_values(
    by=["task", "split", "accuracy"],
    ascending=[True, True, False],
).reset_index(drop=True)

print("\n\n===== Summary metrics table (all tasks / models) =====")
print(results_df_sorted)

# Save overall CSV
overall_csv_path = os.path.join(
    RESULTS_METRICS_DIR,
    "classical_ml_all_tasks_metrics.csv",
)
results_df_sorted.to_csv(overall_csv_path, index=False)
print("\n[INFO] Saved overall metrics CSV to:", overall_csv_path)

# Save per-task CSV for easier report usage / 03 notebook
for task_name in ACTIVE_TASKS:
    task_df = results_df_sorted[results_df_sorted["task"] == task_name]
    task_csv_path = os.path.join(
        RESULTS_METRICS_DIR,
        f"{task_name}_classical_ml_metrics.csv",
    )
    task_df.to_csv(task_csv_path, index=False)
    print(f"[INFO] Saved metrics CSV for task={task_name} to:", task_csv_path)

# ==========================================
# Train/Val 비교 및 과대적합 진단 CSV 생성
# ==========================================

# Train/Val 비교표 생성
train_val_comparison = []
for task_name in ACTIVE_TASKS:
    for model_name in ACTIVE_MODELS:
        task_model_df = results_df_sorted[
            (results_df_sorted["task"] == task_name) & 
            (results_df_sorted["model"] == model_name)
        ]
        
        train_row = task_model_df[task_model_df["split"] == "train"]
        val_row = task_model_df[task_model_df["split"] == "val"]
        
        if len(train_row) > 0 and len(val_row) > 0:
            task_desc_val = train_row.iloc[0].get("task_desc", "") if "task_desc" in train_row.columns else ""
            train_val_comparison.append({
                "task": task_name,
                "task_desc": task_desc_val,
                "model": model_name,
                "train_accuracy": train_row.iloc[0]["accuracy"],
                "val_accuracy": val_row.iloc[0]["accuracy"],
                "train_f1": train_row.iloc[0]["f1"],
                "val_f1": val_row.iloc[0]["f1"],
                "gap": val_row.iloc[0].get("gap", train_row.iloc[0]["accuracy"] - val_row.iloc[0]["accuracy"]),
            })

train_val_df = pd.DataFrame(train_val_comparison)
train_val_df = train_val_df.sort_values(
    by=["task", "val_accuracy"],
    ascending=[True, False]
).reset_index(drop=True)

train_val_csv_path = os.path.join(
    RESULTS_METRICS_DIR,
    "train_val_comparison.csv",
)
train_val_df.to_csv(train_val_csv_path, index=False)
print(f"\n[INFO] Saved train/val comparison CSV to: {train_val_csv_path}")

# 과대적합 진단표 생성
overfitting_diagnosis = []
for task_name in ACTIVE_TASKS:
    for model_name in ACTIVE_MODELS:
        task_model_df = results_df_sorted[
            (results_df_sorted["task"] == task_name) & 
            (results_df_sorted["model"] == model_name)
        ]
        
        val_row = task_model_df[task_model_df["split"] == "val"]
        if len(val_row) > 0:
            gap = val_row.iloc[0].get("gap", 0)
            diagnosis = ""
            if gap > 0.05:
                diagnosis = "과대적합"
            elif gap < 0.01:
                diagnosis = "적절한 적합"
            else:
                diagnosis = "약간의 과대적합"
            
            overfitting_diagnosis.append({
                "task": task_name,
                "model": model_name,
                "train_val_gap": gap,
                "diagnosis": diagnosis,
                "val_accuracy": val_row.iloc[0]["accuracy"],
            })

overfitting_df = pd.DataFrame(overfitting_diagnosis)
overfitting_df = overfitting_df.sort_values(
    by=["task", "train_val_gap"],
    ascending=[True, False]
).reset_index(drop=True)

overfitting_csv_path = os.path.join(
    RESULTS_METRICS_DIR,
    "overfitting_diagnosis.csv",
)
overfitting_df.to_csv(overfitting_csv_path, index=False)
print(f"[INFO] Saved overfitting diagnosis CSV to: {overfitting_csv_path}")

print("\n[INFO] Classical ML training & evaluation pipeline finished.")

## # PART 3: 평가 및 시각화

03_evaluation_and_plots.ipynb에서 가져온 코드입니다.

In [ ]:
# ================================================================
# 03. Error Analysis & Visualization on Colored MNIST
# ================================================================
# 이 노트북은 학습된 모델의 성능을 상세히 분석하고 시각화합니다.
# 
# 주요 기능:
#   1. 전처리된 Colored MNIST 데이터 로드 (train/val만 사용)
#   2. 각 task(digit/fg/bg)에 대해 선택한 모델 학습
#   3. 오분류 분석:
#      - Confusion Matrix (혼동 행렬)
#      - 클래스별 정확도 / 오류율
#      - 데이터 출처별 오류 (raw/deskew/font/geom)
#      - 오분류된 샘플 시각화
#   4. 결과를 CSV와 이미지로 저장

# ================================
# Cell 1. Imports & global config
# ================================
# 이 셀은 노트북 실행에 필요한 모든 라이브러리를 임포트하고,
# Config 파일을 로드하는 유틸리티 함수를 정의합니다.

# ===== 표준 라이브러리 =====
import os  # 운영체제 인터페이스 (파일/디렉토리 경로 조작)
import random  # 랜덤 시드 설정
from pathlib import Path  # 경로 객체를 다루는 모던한 방법
import yaml  # YAML 파일 파싱 (config 파일 읽기용)

# ===== 데이터 처리 라이브러리 =====
import numpy as np  # 수치 연산 및 배열 처리
import pandas as pd  # 데이터프레임 처리 (통계 분석, CSV 저장용)

# ===== 시각화 라이브러리 =====
import matplotlib.pyplot as plt  # 그래프/이미지 시각화
import seaborn as sns  # 통계 시각화 (heatmap 등)

# ===== 머신러닝 라이브러리 =====
from sklearn.model_selection import train_test_split  # 데이터 분할 (사용 안 함, 이미 분할됨)
from sklearn.preprocessing import StandardScaler  # 특성 스케일링
from sklearn.pipeline import Pipeline  # 전처리+모델 파이프라인 구성
import joblib  # 모델 저장/로드용

# ===== 평가 지표 =====
from sklearn.metrics import (
    accuracy_score,  # 정확도 계산
    precision_recall_fscore_support,  # Precision, Recall, F1-score 계산
    classification_report,  # 분류 리포트 출력
    confusion_matrix,  # 혼동 행렬 생성
)

# ===== 분류 모델들 =====
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀
from sklearn.neighbors import KNeighborsClassifier  # K-최근접 이웃
from sklearn.svm import SVC  # 서포트 벡터 머신
from sklearn.tree import DecisionTreeClassifier  # 결정 트리
from sklearn.ensemble import RandomForestClassifier  # 랜덤 포레스트

# XGBoost (선택적; 설치되어 있지 않으면 !pip install xgboost 실행 필요)
from xgboost import XGBClassifier

# ===== Jupyter Notebook 설정 =====
# %matplotlib inline   # 주피터 노트북에서 그래프를 인라인으로 표시 (필요시 주석 해제)

# ===== 재현성을 위한 시드 설정 =====
RANDOM_SEED = 0  # 모든 랜덤 연산의 시드 고정
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ===== Matplotlib 설정 =====
plt.rcParams["font.family"] = "DejaVu Sans"  # 폰트 설정 (한글 깨짐 방지)
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

# ======================================
# Config 로더 함수들
# ======================================
# 이 함수들은 YAML 형식의 config 파일을 읽어서 Python 딕셔너리로 변환합니다.
# 프로젝트의 경로와 설정을 중앙에서 관리하기 위해 사용됩니다.

def load_config(config_path):
    """
    YAML config 파일을 로드하여 dict 반환
    
    역할:
    - configs/ 디렉토리의 YAML 파일을 읽어서 Python 딕셔너리로 변환
    - 노트북이 어디서 실행되든 올바른 경로를 찾아서 config 파일 로드
    
    사용 예시:
        config = load_config("configs/paths.yaml")
        # → {'paths': {'data': {...}, 'results': {...}}}
    """
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    config_file = Path(config_path)
    if not config_file.is_absolute():
        config_file = project_root / config_path
    
    with open(config_file, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def get_paths(config_path="configs/paths.yaml"):
    """
    경로 config를 로드하고 절대 경로로 변환
    
    역할:
    - paths.yaml에서 경로 정보를 읽어서 절대 경로 객체(Path)로 변환
    - 모든 노트북에서 일관된 경로를 사용할 수 있도록 함
    
    사용 예시:
        paths = get_paths()
        npz_path = paths["data"]["processed"]
    """
    config = load_config(config_path)
    paths = config['paths']
    
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    resolved = {}
    for key, value in paths.items():
        if isinstance(value, dict):
            # None 값 처리: v가 None이면 None을 그대로 유지, 아니면 절대 경로로 변환
            resolved[key] = {
                k: project_root / v if v is not None else None 
                for k, v in value.items()
            }
        else:
            # None 값 처리: value가 None이면 None을 그대로 유지
            resolved[key] = project_root / value if value is not None else None
    
    return resolved

print("[INFO] Libraries imported.")

In [ ]:
# ===================================
# Cell 2. Paths & load preprocessed npz (Config-based)
# ===================================
# 이 셀은 데이터 경로와 오분류 분석 결과 저장 경로를 설정하고,
# 전처리된 데이터를 로드합니다.

# ===== Config에서 경로 로드 =====
# get_paths() 함수를 사용하여 configs/paths.yaml의 모든 경로를 딕셔너리로 가져옵니다.
paths = get_paths("configs/paths.yaml")

# ===== 프로젝트 루트 디렉토리 =====
# 현재 작업 디렉토리를 확인하고, 프로젝트 루트를 찾습니다.
BASE_DIR = Path.cwd()
if 'notebooks' in str(BASE_DIR):
    BASE_DIR = BASE_DIR.parent  # notebooks/의 부모 = 프로젝트 루트

# ===== 입력 데이터 경로 =====
# 01_preprocessing_colored_mnist.ipynb에서 생성한 전처리된 데이터셋 경로
NPZ_PATH = paths["data"]["processed"]
# 예: data/processed/colored_mnist/colored_mnist_100k_train_val.npz

# ===== 결과 저장 경로 =====
# 오분류 분석 결과를 저장할 디렉토리들
RESULTS_ROOT_DIR = BASE_DIR / "results"  # 결과 루트 디렉토리
RESULTS_ANALYSIS_DIR = paths["results"]["error_analysis"]  # 오분류 분석 결과 디렉토리
FIGURES_DIR = RESULTS_ANALYSIS_DIR / "figures"  # 시각화 이미지 저장 경로
TABLES_DIR = RESULTS_ANALYSIS_DIR / "tables"  # CSV 테이블 저장 경로
# 예: results/error_analysis/figures/, results/error_analysis/tables/

# 디렉토리가 없으면 자동으로 생성
os.makedirs(RESULTS_ANALYSIS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# ===== 이미지 크기 상수 =====
# Colored MNIST 이미지의 크기 (높이, 너비, 채널)
IMG_SHAPE = (28, 28, 3)  # 28x28 픽셀, RGB 3채널

# ===== 정보 출력 =====
print("[INFO] BASE_DIR          :", BASE_DIR)
print("[INFO] NPZ_PATH          :", NPZ_PATH)
print("[INFO] ANALYSIS_DIR      :", RESULTS_ANALYSIS_DIR)

if not os.path.exists(NPZ_PATH):
    raise FileNotFoundError(f"[ERROR] Preprocessed npz not found at: {NPZ_PATH}")

data = np.load(NPZ_PATH)

X_train = data["X_train"]  # (N_train, 2352)
X_val   = data["X_val"]    # (N_val, 2352)

y_digit_train = data["y_digit_train"]
y_digit_val   = data["y_digit_val"]

y_fg_train = data["y_fg_train"]
y_fg_val   = data["y_fg_val"]

y_bg_train = data["y_bg_train"]
y_bg_val   = data["y_bg_val"]

source_train = data["source_train"]  # raw / deskew / font / geom
source_val   = data["source_val"]

COLOR_PALETTE = data["color_palette"]

print("[INFO] Dataset loaded.")
print("  X_train :", X_train.shape, X_train.dtype)
print("  X_val   :", X_val.shape)
print("  y_digit :", y_digit_train.shape, y_digit_val.shape)
print("  y_fg    :", y_fg_train.shape, y_fg_val.shape)
print("  y_bg    :", y_bg_train.shape, y_bg_val.shape)
print("  source_train/val:", source_train.shape, source_val.shape)

assert X_train.shape[1] == 28 * 28 * 3
assert len(X_train) == len(y_digit_train) == len(y_fg_train) == len(y_bg_train)
assert len(X_val)   == len(y_digit_val)   == len(y_fg_val)   == len(y_bg_val)

In [ ]:
# ==========================================
# Cell 3. Model definitions & Task selection (Config-based)
# ==========================================
# 이 셀은 configs/models.yaml에서 모델 설정을 로드하고,
# 실행할 모델과 태스크를 선택합니다.

# ===== Config에서 모델 설정 로드 =====
from pathlib import Path
import yaml

def load_model_config(config_path="configs/models.yaml"):
    """모델 config를 로드"""
    notebook_dir = Path.cwd()
    if 'notebooks' in str(notebook_dir):
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir
    
    config_file = project_root / config_path
    with open(config_file, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

model_config = load_model_config("configs/models.yaml")
models_cfg = model_config["models"]

# ===== BASE_MODELS 딕셔너리 생성 =====
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

RANDOM_SEED = 0

BASE_MODELS = {}
for model_name, model_params in models_cfg.items():
    base_params = model_params["base"].copy()
    if model_name in ["logreg", "tree", "rf", "xgb"]:
        base_params["random_state"] = RANDOM_SEED
    
    if model_name == "logreg":
        BASE_MODELS[model_name] = LogisticRegression(**base_params)
    elif model_name == "knn":
        BASE_MODELS[model_name] = KNeighborsClassifier(**base_params)
    elif model_name == "svm":
        BASE_MODELS[model_name] = SVC(**base_params)
    elif model_name == "tree":
        BASE_MODELS[model_name] = DecisionTreeClassifier(**base_params)
    elif model_name == "rf":
        BASE_MODELS[model_name] = RandomForestClassifier(**base_params)
    elif model_name == "xgb":
        BASE_MODELS[model_name] = XGBClassifier(**base_params)

# ===== NEEDS_SCALING 딕셔너리 생성 =====
NEEDS_SCALING = {
    model_name: model_params["needs_scaling"]
    for model_name, model_params in models_cfg.items()
}

# ===== Task 선택 =====
TASK_LABEL_INFO = {
    "digit": {
        "y_train": y_digit_train,
        "y_val": y_digit_val,
        "desc": "Digit classification (0-9)",
    },
    "fg": {
        "y_train": y_fg_train,
        "y_val": y_fg_val,
        "desc": "Foreground color classification (7 classes, ROYG BIV)",
    },
    "bg": {
        "y_train": y_bg_train,
        "y_val": y_bg_val,
        "desc": "Background color classification (7 classes, ROYG BIV)",
    },
}

# ===== 실행할 모델 및 태스크 선택 =====
# 옵션 1: 단일 모델
TARGET_MODEL_NAME = None  # "xgb" 또는 None

# 옵션 2: 모든 모델 (배치 실행)
TARGET_MODELS = ["logreg", "knn", "svm", "tree", "rf", "xgb"]  # 또는 None

# 실행할 태스크 선택
ACTIVE_TASKS = ["digit", "fg", "bg"]  # 모든 task 실행

# 모델 선택 로직
if TARGET_MODEL_NAME:
    models_to_run = [TARGET_MODEL_NAME]
elif TARGET_MODELS:
    models_to_run = TARGET_MODELS
else:
    models_to_run = ["xgb"]  # 기본값

print("[INFO] Models to run:", models_to_run)
print("[INFO] Active tasks:", ACTIVE_TASKS)
print("[INFO] Models loaded from config:", list(BASE_MODELS.keys()))


In [ ]:
# ==========================================
# Cell 4. Helper: build pipeline & metrics
# ==========================================
def build_pipeline(model_name: str):
    """
    Create sklearn Pipeline with optional StandardScaler.
    """
    if model_name not in BASE_MODELS:
        raise ValueError(f"Unknown model_name: {model_name}")

    base_estimator = BASE_MODELS[model_name]

    if NEEDS_SCALING.get(model_name, False):
        steps = [("scaler", StandardScaler()), ("clf", base_estimator)]
    else:
        steps = [("clf", base_estimator)]
    return Pipeline(steps)


def compute_metrics(y_true, y_pred, average="macro"):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return acc, prec, rec, f1

In [ ]:
# ==========================================
# Cell 5. Visualization helpers (EDA)
# ==========================================
def reconstruct_images_from_flat(X_flat):
    """
    X_flat: (N, 2352) -> (N, 28, 28, 3)
    """
    N = X_flat.shape[0]
    return X_flat.reshape(N, *IMG_SHAPE)


def plot_sample_images(X_flat, y, title, n_samples=16):
    """
    Random sample 시각화 (colored 이미지)
    """
    n_samples = min(n_samples, X_flat.shape[0])
    idxs = np.random.choice(X_flat.shape[0], size=n_samples, replace=False)
    imgs = reconstruct_images_from_flat(X_flat[idxs])

    cols = min(8, n_samples)
    rows = int(np.ceil(n_samples / cols))

    plt.figure(figsize=(cols * 1.5, rows * 1.5))
    for i, (img, idx) in enumerate(zip(imgs, idxs)):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img.astype(np.uint8))
        ax.axis("off")
        ax.set_title(str(int(y[idx])), fontsize=8)
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_label_distribution_simple(y_train, y_val, task_name):
    """
    Train/Val label distribution (간단 버전)
    """
    def counts(y):
        return pd.Series(y).value_counts().sort_index()

    train_c = counts(y_train)
    val_c   = counts(y_val)

    df = pd.DataFrame({"train": train_c, "val": val_c}).fillna(0).astype(int)
    df.index.name = "label"

    print(f"\n[EDA] Label distribution for task = {task_name}")
    print(df)

    x = np.arange(len(df.index))
    width = 0.4

    plt.figure(figsize=(8, 4))
    plt.bar(x - width/2, df["train"], width=width, label="train")
    plt.bar(x + width/2, df["val"],   width=width, label="val")
    plt.xticks(x, df.index)
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.title(f"Label distribution (train/val) - {task_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# Cell 6. Confusion matrix & error plots
# ==========================================
def plot_confusion_matrix(y_true, y_pred, classes, title, save_path=None, normalize=False):
    """
    Confusion matrix (optional normalized).
    """
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    if normalize:
        cm = cm.astype(float)
        cm = cm / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f" if normalize else "d",
        xticklabels=classes,
        yticklabels=classes,
        cmap="YlGnBu",
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)
        print("[INFO] Saved confusion matrix to:", save_path)

    plt.show()


def plot_class_accuracy_bar(df_summary, title, save_path=None):
    """
    df_summary: index = label, columns = ['n_true', 'n_correct', 'acc']
    """
    plt.figure(figsize=(8, 4))
    sns.barplot(
        x=df_summary.index.astype(str),
        y=df_summary["acc"],
    )
    plt.ylim(0, 1.0)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)
        print("[INFO] Saved class accuracy barplot to:", save_path)

    plt.show()


def plot_source_accuracy_bar(df_source, title, save_path=None):
    """
    df_source: index = source(raw/deskew/font/geom), columns=['n', 'acc']
    """
    plt.figure(figsize=(6, 4))
    sns.barplot(
        x=df_source.index,
        y=df_source["acc"],
    )
    plt.ylim(0, 1.0)
    plt.xlabel("Source")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200)
        print("[INFO] Saved source accuracy barplot to:", save_path)

    plt.show()

In [ ]:
# ==========================================
# Cell 7. Misclassified sample visualization
# ==========================================
def show_misclassified_examples(
    X_val,
    df_errors,
    task_name,
    n=16,
    random_state=0,
):
    """
    df_errors: DataFrame with columns [idx, y_true, y_pred, source, ...]
    """
    if df_errors.empty:
        print(f"[INFO] No misclassified samples for task={task_name}.")
        return

    rng = np.random.default_rng(random_state)
    n = min(n, len(df_errors))
    chosen = df_errors.sample(n=n, random_state=random_state)

    imgs = reconstruct_images_from_flat(X_val[chosen["idx"].values])

    cols = min(8, n)
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(cols * 1.8, rows * 1.8))
    for i, (idx_row, row) in enumerate(chosen.iterrows()):
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(imgs[i].astype(np.uint8))
        ax.axis("off")
        ax.set_title(
            f"T:{row['y_true']} / P:{row['y_pred']}\nsrc:{row['source']}",
            fontsize=7,
        )
    plt.suptitle(f"Misclassified examples ({task_name})", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# Cell 8. Main error-analysis loop (배치 실행)
# ==========================================
all_task_summary = []

# 모든 모델 × 모든 task에 대해 실행
for model_name in models_to_run:
    print(f"\n\n{'='*60}")
    print(f"### MODEL: {model_name}")
    print(f"{'='*60}")
    
    for task_name in ACTIVE_TASKS:
        info = TASK_LABEL_INFO[task_name]
        y_train_task = info["y_train"]
        y_val_task   = info["y_val"]
        task_desc    = info["desc"]

        print("\n\n##########################################")
        print(f"### Task: {task_name} - {task_desc}")
        print("##########################################")

        # ---- 1) EDA: label distribution + 샘플 이미지 ----
        plot_label_distribution_simple(y_train_task, y_val_task, task_name)
        plot_sample_images(X_val, y_val_task, title=f"Val samples ({task_name})", n_samples=16)

            # ---- 2) 모델 학습 (train -> val) ----
        print(f"\n[TRAIN] Building pipeline for model = {model_name}")
        pipe = build_pipeline(model_name)

        print("[TRAIN] Fitting model...")
        pipe.fit(X_train, y_train_task)

        print("[VAL] Predicting...")
        y_val_pred = pipe.predict(X_val)

        acc, prec, rec, f1 = compute_metrics(y_val_task, y_val_pred, average="macro")
        print(f"\n[VAL] Macro metrics ({task_name}, model={model_name})")
        print(f"  accuracy : {acc:.4f}")
        print(f"  precision: {prec:.4f}")
        print(f"  recall   : {rec:.4f}")
        print(f"  f1-score : {f1:.4f}")

        print("\n[VAL] classification_report:")
        print(classification_report(y_val_task, y_val_pred, digits=4))

        # ---- 3) Confusion matrix (raw + normalized) ----
        classes = np.sort(np.unique(y_val_task))

        cm_title = f"{task_name} - {model_name} (Val, count)"
        cm_path  = os.path.join(FIGURES_DIR, f"cm_{task_name}_{model_name}_val_counts.png")
        plot_confusion_matrix(
            y_true=y_val_task,
            y_pred=y_val_pred,
            classes=classes,
            title=cm_title,
            save_path=cm_path,
            normalize=False,
        )

        cmn_title = f"{task_name} - {model_name} (Val, normalized)"
        cmn_path  = os.path.join(FIGURES_DIR, f"cm_{task_name}_{model_name}_val_normalized.png")
        plot_confusion_matrix(
            y_true=y_val_task,
            y_pred=y_val_pred,
            classes=classes,
            title=cmn_title,
            save_path=cmn_path,
            normalize=True,
        )

        # ---- 4) Per-class accuracy / error rate ----
        df_val = pd.DataFrame({
            "idx": np.arange(len(y_val_task)),
            "y_true": y_val_task,
            "y_pred": y_val_pred,
            "source": source_val,
        })

        # class-wise summary
        class_summary = (
            df_val
            .groupby("y_true")
            .apply(lambda g: pd.Series({
                "n_true": len(g),
                "n_correct": (g["y_true"] == g["y_pred"]).sum(),
            }))
        )
        class_summary["acc"] = class_summary["n_correct"] / class_summary["n_true"]

        print(f"\n[VAL] Class-wise accuracy summary ({task_name}):")
        print(class_summary)

        # 저장
        class_csv_path = os.path.join(
            TABLES_DIR, f"class_accuracy_{task_name}_{model_name}.csv"
        )
        class_summary.to_csv(class_csv_path)
        print("[INFO] Saved class-wise accuracy table to:", class_csv_path)

        # 시각화
        class_plot_title = f"{task_name} - Class-wise accuracy (Val, model={model_name})"
        class_plot_path  = os.path.join(
            FIGURES_DIR, f"class_accuracy_{task_name}_{model_name}.png"
        )
        plot_class_accuracy_bar(class_summary, class_plot_title, class_plot_path)

        # ---- 5) Source-wise accuracy (raw / deskew / font / geom) ----
        source_summary = (
            df_val
            .groupby("source")
            .apply(lambda g: pd.Series({
                "n": len(g),
                "n_correct": (g["y_true"] == g["y_pred"]).sum(),
            }))
        )
        source_summary["acc"] = source_summary["n_correct"] / source_summary["n"]

        print(f"\n[VAL] Source-wise accuracy summary ({task_name}):")
        print(source_summary)

        source_csv_path = os.path.join(
            TABLES_DIR, f"source_accuracy_{task_name}_{model_name}.csv"
        )
        source_summary.to_csv(source_csv_path)
        print("[INFO] Saved source-wise accuracy table to:", source_csv_path)

        source_plot_title = f"{task_name} - Source-wise accuracy (Val, model={model_name})"
        source_plot_path  = os.path.join(
            FIGURES_DIR, f"source_accuracy_{task_name}_{model_name}.png"
        )
        plot_source_accuracy_bar(source_summary, source_plot_title, source_plot_path)

        # ---- 6) Misclassified samples df + 저장 ----
        df_errors = df_val[df_val["y_true"] != df_val["y_pred"]].copy()
        print(f"\n[VAL] Misclassified samples count ({task_name}): {len(df_errors)} "
              f" / {len(df_val)} (error rate={len(df_errors)/len(df_val):.4f})")

        # 가장 많이 헷갈리는 (true,pred) 조합 상위 몇 개
        pair_counts = (
            df_errors
            .groupby(["y_true", "y_pred"])
            .size()
            .sort_values(ascending=False)
        )
        print(f"\n[VAL] Top (true -> pred) confusion pairs ({task_name}):")
        print(pair_counts.head(10))

        # 저장
        error_csv_path = os.path.join(
            TABLES_DIR, f"errors_{task_name}_{model_name}.csv"
        )
        df_errors.to_csv(error_csv_path, index=False)
        print("[INFO] Saved misclassified samples table to:", error_csv_path)

        # ---- 7) Misclassified sample visualization ----
        show_misclassified_examples(
            X_val=X_val,
            df_errors=df_errors,
            task_name=task_name,
            n=16,
            random_state=RANDOM_SEED,
        )

        # ---- 8) 전체 요약 저장용 ----
        all_task_summary.append({
            "task": task_name,
            "task_desc": task_desc,
            "model": model_name,
            "n_val": len(df_val),
            "n_errors": len(df_errors),
            "error_rate": len(df_errors) / len(df_val),
            "acc_macro": acc,
            "precision_macro": prec,
            "recall_macro": rec,
            "f1_macro": f1,
        })

print("\n[INFO] Error analysis finished for all tasks.")

In [ ]:
# ==========================================
# Cell 9. Global summary table
# ==========================================
summary_df = pd.DataFrame(all_task_summary)
summary_df = summary_df.sort_values(by="task").reset_index(drop=True)

print("\n===== Error-analysis summary across tasks =====")
print(summary_df)

summary_csv_path = os.path.join(
    TABLES_DIR, f"error_analysis_summary_all_models.csv"
)
summary_df.to_csv(summary_csv_path, index=False)
print("\n[INFO] Saved global error-analysis summary to:", summary_csv_path)

In [ ]:
# ==========================================
# Cell 10. Test Set Evaluation (교수님 Test 셋 평가)
# ==========================================
# 이 셀은 교수님께서 제공하신 Test 셋을 평가합니다.
# Test 셋 경로는 configs/paths.yaml의 paths.data.test에 설정되어 있습니다.

# ===== Test 셋 경로 확인 =====
TEST_NPZ_PATH = paths["data"].get("test", None)

if TEST_NPZ_PATH and TEST_NPZ_PATH is not None and os.path.exists(TEST_NPZ_PATH):
    print(f"[INFO] Test 셋 경로: {TEST_NPZ_PATH}")
    
    # Test 셋 로드
    test_data = np.load(TEST_NPZ_PATH)
    X_test = test_data["X_test"]
    y_digit_test = test_data.get("y_digit_test", None)
    y_fg_test = test_data.get("y_fg_test", None)
    y_bg_test = test_data.get("y_bg_test", None)
    
    print(f"[INFO] Test 셋 로드 완료: X_test shape = {X_test.shape}")
    
    # 각 task별 평가
    test_results = []
    
    for task_name in ACTIVE_TASKS:
        # Task별 라벨 선택
        y_test_task = {
            "digit": y_digit_test,
            "fg": y_fg_test,
            "bg": y_bg_test
        }[task_name]
        
        if y_test_task is None:
            print(f"[WARN] Test 셋에 {task_name} 라벨이 없습니다. 건너뜁니다.")
            continue
        
        print(f"\n\n{'='*60}")
        print(f"### Test Set Evaluation: {task_name}")
        print(f"{'='*60}")
        
        # 각 모델별 평가
        for model_name in models_to_run:
            # 모델 로드 (02 노트북에서 저장한 모델)
            model_path = os.path.join(paths["results"]["models"], f"{task_name}_{model_name}.joblib")
            
            if not os.path.exists(model_path):
                print(f"[WARN] 모델 파일이 없습니다: {model_path}")
                print(f"[INFO] 모델을 새로 학습합니다...")
                # 모델 재학습
                info = TASK_LABEL_INFO[task_name]
                y_train_task = info["y_train"]
                pipe = build_pipeline(model_name)
                pipe.fit(X_train, y_train_task)
            else:
                print(f"[INFO] 모델 로드: {model_path}")
                pipe = joblib.load(model_path)
            
            # Test 셋 예측
            y_test_pred = pipe.predict(X_test)
            
            # Test 셋 성능 평가
            test_acc, test_prec, test_rec, test_f1 = compute_metrics(y_test_task, y_test_pred, average="macro")
            
            print(f"\n[TEST] Macro metrics ({task_name}, model={model_name})")
            print(f"  accuracy : {test_acc:.4f}")
            print(f"  precision: {test_prec:.4f}")
            print(f"  recall   : {test_rec:.4f}")
            print(f"  f1-score : {test_f1:.4f}")
            
            # Test 셋 Confusion Matrix
            classes = np.sort(np.unique(y_test_task))
            cm_test_title = f"{task_name} - {model_name} (Test, count)"
            cm_test_path = os.path.join(FIGURES_DIR, f"cm_{task_name}_{model_name}_test_counts.png")
            plot_confusion_matrix(
                y_true=y_test_task,
                y_pred=y_test_pred,
                classes=classes,
                title=cm_test_title,
                save_path=cm_test_path,
                normalize=False,
            )
            
            cm_test_norm_title = f"{task_name} - {model_name} (Test, normalized)"
            cm_test_norm_path = os.path.join(FIGURES_DIR, f"cm_{task_name}_{model_name}_test_normalized.png")
            plot_confusion_matrix(
                y_true=y_test_task,
                y_pred=y_test_pred,
                classes=classes,
                title=cm_test_norm_title,
                save_path=cm_test_norm_path,
                normalize=True,
            )
            
            # Test 셋 결과 저장
            test_results.append({
                "task": task_name,
                "model": model_name,
                "test_accuracy": test_acc,
                "test_precision": test_prec,
                "test_recall": test_rec,
                "test_f1": test_f1,
            })
            
            # Test 셋 오분류 분석
            df_test = pd.DataFrame({
                "idx": np.arange(len(y_test_task)),
                "y_true": y_test_task,
                "y_pred": y_test_pred,
            })
            
            df_test_errors = df_test[df_test["y_true"] != df_test["y_pred"]].copy()
            error_csv_path = os.path.join(
                TABLES_DIR, f"test_errors_{task_name}_{model_name}.csv"
            )
            df_test_errors.to_csv(error_csv_path, index=False)
            print(f"[INFO] Test 셋 오분류 분석 저장: {error_csv_path}")
    
    # Test 셋 결과 요약 저장
    if test_results:
        test_results_df = pd.DataFrame(test_results)
        test_results_df = test_results_df.sort_values(
            by=["task", "test_accuracy"],
            ascending=[True, False]
        ).reset_index(drop=True)
        
        test_summary_csv_path = os.path.join(
            TABLES_DIR, "test_metrics_all_models.csv"
        )
        test_results_df.to_csv(test_summary_csv_path, index=False)
        print(f"\n[INFO] Test 셋 결과 요약 저장: {test_summary_csv_path}")
        print("\n===== Test Set Results Summary =====")
        print(test_results_df)
    
else:
    print("[INFO] Test 셋이 없습니다.")
    print("[INFO] Test 셋 경로를 configs/paths.yaml의 paths.data.test에 설정해주세요.")
    print("[INFO] 예: test: \"data/test/professor_test_set.npz\"")
